# MRM-Bench baseline: logistic regression, step by step

## 0. What this notebook does, and how to use it

**What it is.** This notebook builds the clean baseline model of MRM-Bench. It is the only implementation of the
baseline. Every step is explained, and every step shows its result.

**The question the model answers.** Take every mortgage loan that, at the end of a December, is still active and
not seriously late. What is the chance that it becomes **90 or more days late** at least once in the next 12 months?
The model may use only what was known on that 31 December.

**A small example.** On 31 December 2017 a loan is 30 days late. Will it be 90+ days late in any month from January
to December 2018? The model answers with a probability, for example 2%.

This outcome is "12-month 90+ DPD serious delinquency" (DPD = days past due). It is **not** a regulatory default
definition. The full rules are in `CLAUDE.md`, section "Outcome specification" (a draft, waiting for Dr. Ku's
approval).

**How we know the results are right.** The official results are stored in `reference/`. Section 12 compares every
result of this notebook with them and must print `MATCHES THE REFERENCE MODEL`. If you change something, section 12
lists exactly which results changed.

**History.** The baseline was first written as a script, `src/baseline_logreg.py`. That script made the files in
`reference/`. It is no longer in the repository, but git keeps it: `git show 9253e97:src/baseline_logreg.py`. This
notebook uses the same settings, column names, seed and feature lists. The script's functions that run only once
(`open_files()`, `load()`, `origination_features()`, `calibration_table()`) are written out here as separate small
cells, so you can see the result of each step; `loan_pool()` is now called `loan_is_in_train_pool()`. The functions
that run many times (`build_snapshot()`, `evaluate()`, `cluster_bootstrap_ci()`) are still functions.

### Words used in this notebook

| word | meaning |
|---|---|
| origination data | facts about a loan on the day it was made (credit score, loan amount, ...): one row per loan |
| performance data | the loan's monthly history (how late, balance, payoff, ...): one row per loan per month |
| delinquency status | how late the payments are: `00` = on time (or less than 30 days late), `01` = 30–59 days late, `02` = 60–89 days, `03` = 90–119 days, and so on |
| 90+ | 90 or more days late (status `03` or higher) |
| REO | "real estate owned": the lender has taken the house (status `RA`) |
| prediction month $t$ | the December at which we predict (for example December 2017) |
| at risk | the loan is still active at $t$ and not yet 90+ late, so it can still *become* 90+ |
| event (y = 1) | the loan is 90+ late or REO in at least one of the 12 months after $t$ |
| non-event (y = 0) | no event, and we still see the loan 12 months after $t$ |
| censored | we stopped seeing the loan before the 12 months ended (for example, it was paid off), so we do not know its outcome |
| snapshot | a table with one row per at-risk loan for one December |
| feature | an input of the model (for example the credit score) |
| leakage | a feature that uses information from after $t$; it makes the model look better than it can really be |

### How to use it

1. **Choose the kernel (VS Code).** Click **Select Kernel** (top right) → **Python Environments…** → pick
   **finance-ai-system** or **.venv (Python 3.12.12)**, path `.venv/bin/python`. Not in the list? Run `uv sync` in
   a terminal at the repository root, then run the command **Developer: Reload Window**. The imports cell in
   section 1 prints the kernel path, so you can check.
2. **Run from top to bottom** with **Run All**. Later cells use variables made by earlier cells. After you change a
   setting, rerun from the section given in the table in section 13, or simply **Restart** and **Run All**.
3. **Run time:** about 1 minute (measured on the development Mac: 53 seconds for the whole notebook, 37 of them for
   the bootstrap in section 9). For a quick run, set `N_BOOT = 100` in the SETTINGS cell;
   section 12 will then report that only the confidence intervals differ.
4. **Files:** the notebook reads `data/raw/sample_2015.zip` and `reference/`. It unzips the data into
   `sflld_extracted/` (gitignored). Only when `SAVE_OUTPUTS = True` does it write five result files to `outputs/`
   (gitignored). It never writes to `reference/`.

> **Warning: real loan data.** This notebook shows real Freddie Mac records (small samples). Freddie Mac's terms do
> not allow sharing them. **nbstripout** removes all cell outputs when you commit, so outputs never enter git. It
> does **not** clean the file on your disk: before you email, upload or share the `.ipynb` file itself, run
> **Clear All Outputs**. nbstripout is set up per clone: after a fresh clone, run `uv sync` and then
> `uv run nbstripout --install`. Check it with `uv run nbstripout --status`.

To run the whole notebook without VS Code (the executed copy is written outside the repository):

```
uv run jupyter nbconvert --to notebook --execute notebooks/baseline_logistic_regression.ipynb --output-dir /tmp/nb_check
```

### Contents

| § | Section | What you learn |
|---|---|---|
| 0 | What this notebook does | this page |
| 1 | Setup and settings | paths, imports, the one **SETTINGS** cell |
| 2 | Loading the raw files | column names, reading everything as text, three checks |
| 3 | Cleaning the origination data | "Not Available" codes, the five derived features |
| 4 | Cleaning the monthly data | the month counter $t$, delinquency and zero-balance codes |
| 5 | A toy example (made-up data) | how `build_snapshot()` makes labels and features, line by line |
| 6 | Building the real snapshots | label counts, the December 2019 spike |
| 7 | The train/test split | split by loan (hash) and by time; why censored rows are dropped |
| 8 | The logistic regression model | the math in plain words, preprocessing, fitting |
| 9 | Evaluation | ROC-AUC, PR-AUC, Brier score, the bootstrap, calibration |
| 10 | Coefficients | how to read them |
| 11 | Leakage check | changing the future must not change any feature |
| 12 | Does the notebook match the reference? | the final verdict |
| 13 | Where to make changes | safe edit points and exercises |

## 1. Setup and settings

**What:** Moves the working folder to the repository root (the folder that contains `pyproject.toml`).

**Why:** VS Code starts the notebook in `notebooks/`, but all paths in this notebook (`data/raw/...`, `reference/`)
start from the repository root.

**Check:** the printed folder ends in `Finance AI System`.

In [ ]:
import os
from pathlib import Path

# Start in the folder where the notebook runs (usually notebooks/).
folder = Path.cwd().resolve()

# Go up one folder at a time until we find pyproject.toml: that folder is the repository root.
while not (folder / "pyproject.toml").exists():
    if folder.parent == folder:
        # We reached the top of the disk and found nothing.
        raise FileNotFoundError("pyproject.toml not found: open the notebook inside the repository")
    folder = folder.parent

# From now on, work from the repository root.
ROOT = folder
os.chdir(ROOT)
print("Working folder:", Path.cwd())

**What:** Imports the libraries, and prints their versions and the path of the Python that runs the notebook (the
*kernel*).

**Why:** The results can only match the reference if the library versions are the same. Section 12 compares them
with the versions stored in `reference/metrics.json`.

**Check:** Python 3.12.12, numpy 2.5.4, pandas 3.1.0, scikit-learn 1.9.1, and a kernel path ending in
`.venv/bin/python`.

In [ ]:
# Standard library (comes with Python)
import hashlib     # SHA-256 hash, for the train/test split (section 7)
import inspect     # reads the source code of a function (leakage demo, section 11)
import io          # lets pandas read a text string as if it were a file (section 2)
import json        # reads and writes metrics.json
import platform    # the Python version
import sys         # the path of the running Python
import time        # measures how long the evaluation takes
import zipfile     # unzips the data

# Tables and numbers
import numpy as np
import pandas as pd

# The model and its preprocessing (scikit-learn)
import sklearn
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, brier_score_loss, roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Plots and table display
import matplotlib.pyplot as plt
from IPython.display import display

print("Python      ", platform.python_version())
print("numpy       ", np.__version__)
print("pandas      ", pd.__version__)
print("scikit-learn", sklearn.__version__)
print("Kernel      ", sys.executable)

**What:** Holds **every setting** of the baseline in one place. This is the only cell where settings are defined.

**Why:** With one settings cell, every experiment is visible here, and section 12 reports which reference results
it changed.

**Check:** nothing is printed. With these default values, section 12 must print `MATCHES THE REFERENCE MODEL`.

In [ ]:
# ===============================================================================================
#  SETTINGS: change things here, then rerun from the section named in the section 13 table
#  (or Restart + Run All).
# ===============================================================================================

# The data: a zip (or a folder) with sample_orig_*.txt and sample_perf_*.txt.
# Another vintage means other loans, so nothing would match the reference.
DATA_PATH = "data/raw/sample_2015.zip"

# Random seed for the bootstrap (section 9). Changing it should change only the 95% confidence intervals.
# (It is also given to LogisticRegression, but its lbfgs solver does not use randomness.)
SEED = 20261022

# Prediction months: Decembers, written YYYYMM.
# TRAIN: used to fit the model. Their 12-month windows end by December 2019, before the pandemic.
TRAIN_SNAPSHOTS = [201512, 201612, 201712, 201812]
# TEST: used to test the model. Their windows end by December 2025 (the data end in March 2026).
# 202512 would need data up to December 2026, so its non-events could not be observed yet.
TEST_SNAPSHOTS = [202112, 202212, 202312, 202412]
# STRESS: the pandemic windows. Reported only, never used to fit.
STRESS_SNAPSHOTS = [201912, 202012]

# Outcome window in months: the label looks at months t+1 .. t+HORIZON.
# Changing it changes every label, so label counts, metrics and coefficients all change.
HORIZON = 12

# Percent of loans (hash buckets 0-99) that may be used for training. The others may be used for test/stress.
TRAIN_LOAN_SHARE = 70

# Inverse strength of the L2 penalty: a smaller C pulls the coefficients more strongly toward 0.
C = 1.0

# Number of bootstrap resamples for the 95% confidence intervals.
# 100 gives a quick run, but the reference used 1000, so any other value changes the intervals.
N_BOOT = 1000

# True = at the end of section 12, also write the five result files to outputs/ (gitignored).
# It changes no result, and reference/ is never written.
SAVE_OUTPUTS = False

## 2. Loading the raw files

`sample_2015.zip` holds two text files. In both, the fields are separated by `|` (a "pipe-delimited" file):

* `sample_orig_2015.txt`: the **origination** data, one row per loan, 31 fields.
* `sample_perf_2015.txt`: the **performance** data, one row per loan per month, 35 fields.

The files have **no header row** (General User Guide, July 2026, p. 4: *"The origination and monthly performance
data files do not contain a header row."*). So we write the column names ourselves, in the order of the July 2026
file layout (`data/raw/file_layout_july_2026.xlsx`). If the order were wrong, every column would silently get the
wrong name. The count check below and the value checks in section 3 (for example, credit scores between 300 and
850) protect against that.

**What:** Lists the 31 origination and the 35 performance column names, in the order of the file layout.

**Why:** The files have no header row, so these lists are the only thing that tells pandas which field is which.

**Check:** no error, and the numbers 31 and 35.

In [ ]:
ORIG_COLS = ["fico", "first_payment_date", "first_time_homebuyer", "maturity_date", "msa", "mi_pct",
             "num_units", "occupancy_status", "orig_cltv", "orig_dti", "orig_upb", "orig_ltv",
             "orig_interest_rate", "channel", "ppm_flag", "amortization_type", "property_state",
             "property_type", "postal_code", "loan_id", "loan_purpose", "orig_loan_term",
             "num_borrowers", "seller_name", "super_conforming_flag", "pre_harp_loan_id",
             "special_eligibility_program", "harp_indicator", "property_valuation_method",
             "io_indicator", "vantagescore4"]

PERF_COLS = ["loan_id", "period", "current_upb", "delinq_status", "loan_age", "rem_months_to_maturity",
             "defect_settlement_date", "modification_flag", "zero_balance_code", "zero_balance_date",
             "current_interest_rate", "current_non_int_upb", "ddlpi", "mi_recoveries",
             "net_sales_proceeds", "non_mi_recoveries", "total_expenses", "legal_costs",
             "maint_pres_costs", "taxes_insurance", "misc_expenses", "actual_loss", "cum_mod_costs",
             "step_mod_flag", "payment_deferral_flag", "eltv", "zb_removal_upb",
             "delinq_accrued_interest", "delinq_due_to_disaster", "borrower_assistance_plan",
             "current_period_mod_costs", "current_int_upb", "mi_cancellation_flag", "servicer_name",
             "bankruptcy_cramdown_costs"]

# The documented number of fields in each file.
assert len(ORIG_COLS) == 31
assert len(PERF_COLS) == 35
print(len(ORIG_COLS), len(PERF_COLS))

**Why we read every field as text.** The files contain codes, not only numbers: delinquency status `00`, `01`,
`RA`; zero-balance code `01`; credit score `9999` = "Not Available". If pandas guesses the types itself:

* `00` becomes the number 0 and `01` becomes 1, so a later test `== "01"` silently finds nothing;
* `9999` looks like a real (but impossible) credit score;
* the text `NA` becomes a missing value.

So we read every column as text (`dtype=str`), and we tell pandas that only an empty field is missing
(`keep_default_na=False`, `na_values=[""]`). Sections 3 and 4 then convert each code on purpose.

**What:** Reads one made-up line twice: once with pandas' default type guessing, once with our text settings.

**Why:** It shows, on a harmless line, what the text settings protect against.

**Check:** in the "default" column, `00` → 0, `01` → 1 and `NA` → NaN (missing). In the "as text" column every code
stays exactly as written.

In [ ]:
# A made-up record: loan, month, status, zero-balance code, credit-score code, and the text "NA".
toy_line = "X01|201712|00|01|9999|NA\n"

# io.StringIO lets pandas read a string as if it were a file.
default_read = pd.read_csv(io.StringIO(toy_line), sep="|", header=None)
text_read = pd.read_csv(io.StringIO(toy_line), sep="|", header=None,
                        dtype=str, keep_default_na=False, na_values=[""])

# Show the two readings side by side: one row per field.
comparison = pd.DataFrame({
    "default parsing": default_read.iloc[0],
    "as text (our settings)": text_read.iloc[0],
})
display(comparison)

**What:** Unzips the data into `sflld_extracted/` and finds the two text files.

**Why:** `DATA_PATH` may point at the zip or at a folder that is already unzipped; both work.

**Check:** two file paths, one ending in `sample_orig_2015.txt` and one in `sample_perf_2015.txt`.

In [ ]:
data_path = Path(DATA_PATH)

if data_path.suffix == ".zip":
    # Unzip into sflld_extracted/. That folder is gitignored, so the data never enter git.
    data_folder = Path("sflld_extracted")
    data_folder.mkdir(exist_ok=True)
    with zipfile.ZipFile(data_path) as zip_file:
        zip_file.extractall(data_folder)
else:
    # DATA_PATH is a folder that is already unzipped.
    data_folder = data_path

# Find the two files. rglob also searches sub-folders. We take the first match of each.
orig_matches = list(data_folder.rglob("sample_orig_*.txt"))
perf_matches = list(data_folder.rglob("sample_perf_*.txt"))
orig_file = orig_matches[0]
perf_file = perf_matches[0]

print("Origination file:", orig_file)
print("Performance file:", perf_file)

**What:** Reads both files as text. From the performance file we keep only the 9 fields the baseline uses.

**Why:** This is the starting point for everything else.

**Check:** `Loaded 50,000 loans and 3,467,166 monthly records`. It takes a few seconds.

In [ ]:
# Kept from the original script, so every run starts from the same random state.
# (The bootstrap in section 9 does not use it: it makes its own random generator from SEED.)
np.random.seed(SEED)

# The 9 performance fields the baseline uses (of 35).
PERF_COLS_USED = ["loan_id", "period", "current_upb", "delinq_status", "loan_age", "modification_flag",
                  "zero_balance_code", "delinq_due_to_disaster", "borrower_assistance_plan"]

# Origination file: one row per loan, every field as text.
orig = pd.read_csv(
    orig_file,
    sep="|",                 # fields are separated by |
    header=None,             # the file has no header row ...
    names=ORIG_COLS,         # ... so we give the column names
    dtype=str,               # every field as text
    keep_default_na=False,   # do not turn texts such as "NA" into missing values
    na_values=[""],          # only an empty field is missing
)

# Performance file: one row per loan per month. The same text settings, and only the 9 fields above.
perf = pd.read_csv(
    perf_file,
    sep="|",
    header=None,
    names=PERF_COLS,
    usecols=PERF_COLS_USED,
    dtype=str,
    keep_default_na=False,
    na_values=[""],
)

print(f"Loaded {len(orig):,} loans and {len(perf):,} monthly records")

**What:** Runs three checks on the structure of the data, and prints the numbers behind them.

**Why:** Every later step relies on three facts: (1) one row per loan in the origination data; (2) every monthly
record belongs to a known loan; (3) at most one record per loan per month. If one of them failed, the snapshots
would count loans twice or lose them.

**Check:** 50,000 rows and 50,000 different loan IDs; 0 unknown loans; 0 duplicate loan-months; months from 201501
to 202603 (135 months).

In [ ]:
# Check 1: every loan ID appears only once in the origination data.
n_rows = len(orig)
n_different_ids = orig["loan_id"].nunique()
print(f"Origination: {n_rows:,} rows, {n_different_ids:,} different loan IDs, {orig.shape[1]} columns")
assert orig["loan_id"].is_unique, "duplicate loan IDs in the origination file"

# Check 2: every loan in the monthly data is also in the origination data.
orig_ids = set(orig["loan_id"])
perf_ids = set(perf["loan_id"])
unknown_ids = perf_ids - orig_ids              # loans in perf that are not in orig
print(f"Performance: {len(perf):,} monthly records for {len(perf_ids):,} loans, {perf.shape[1]} columns")
print("Monthly-data loans missing from the origination data:", len(unknown_ids))
assert len(unknown_ids) == 0, "performance loans missing from origination"

# Check 3: at most one record per loan per month.
is_duplicate = perf.duplicated(["loan_id", "period"])
n_duplicates = int(is_duplicate.sum())
print("Duplicate loan-months:", n_duplicates)
assert n_duplicates == 0, "duplicate loan-month records"

# The range of months in the data.
first_month = perf["period"].min()
last_month = perf["period"].max()
n_months = perf["period"].nunique()
print(f"Months: {first_month} to {last_month} ({n_months} different months)")

**What:** Shows the first 5 rows of each table.

**Why:** To see what the raw data look like before any cleaning.

**Check:** every field is still text, for example `00` and `9999`. In the monthly table: the same loan in months that
follow each other.

In [ ]:
display(orig.head(5))    # 5 real loans, all 31 origination fields, still text
display(perf.head(5))    # their first monthly records (9 fields)

## 3. Cleaning the origination data

Some numeric fields use a special value, instead of an empty field, when a value is "Not Available". These values
are **codes, not numbers** (they are also called *sentinels*).

Example: a debt-to-income ratio (DTI) of `999` does not mean a borrower with huge debts. It means the ratio is not
disclosed. Used as a number, 999 would distort the model (benchmark pair 1 plants exactly this defect).

So we first list the documented codes and turn them into missing values (NaN, "not a number", is pandas' marker for
a missing value). Only **after** that do we convert the fields to numbers.

In the original script all steps of this section were one function, `origination_features()`. Here each step is
its own cell.

**What:** Lists, for each numeric field, the code that the guide documents as "Not Available", with its meaning and
page in the guide.

**Why:** These are the only values we turn into missing values, so each one must be backed by the documentation.

**Check:** nothing is printed.

In [ ]:
# Documented "Not Available" codes (General User Guide, July 2026). These codes become NaN.
SENTINELS = {
    "fico": "9999",          # credit score
    "mi_pct": "999",         # mortgage insurance percentage
    "num_units": "99",       # number of units in the property
    "orig_cltv": "999",      # combined loan-to-value ratio
    "orig_dti": "999",       # debt-to-income ratio
    "orig_ltv": "999",       # loan-to-value ratio
    "num_borrowers": "99",   # number of borrowers
}

# What the guide says about each code.
GUIDE_MEANING = {
    "fico": "Not Available (scores < 300 or > 850 are also disclosed as Not Available)",
    "mi_pct": "Not Available (MI < 0% or > 55% is also disclosed as Not Available; 0 = no MI)",
    "num_units": "Not Available (values < 1 or > 4 are also disclosed as Not Available)",
    "orig_cltv": "Not Available (outside the documented range, or CLTV < LTV)",
    "orig_dti": "Not Available (ratios > 65%; ALL HARP loans are disclosed as Not Available)",
    "orig_ltv": "Not Available (outside the documented range)",
    "num_borrowers": "Not Available (values < 1 or > 10 are also disclosed as Not Available)",
}

# The page of the guide for each code (page numbers as printed in the guide).
GUIDE_PAGE = {
    "fico": "4",
    "mi_pct": "4-5",
    "num_units": "5",
    "orig_cltv": "5",
    "orig_dti": "5",
    "orig_ltv": "6",
    "num_borrowers": "8",
}

**What:** Makes a copy of the origination table, called `orig_clean`, and replaces each documented code with NaN.

**Why:** After this step a code can no longer be mistaken for a number. `orig` itself stays exactly as it was read
from the file, so we can compare before and after.

**Check:** nothing is printed; the next cell shows the counts.

In [ ]:
# Work on a copy, so that orig keeps the raw text.
orig_clean = orig.copy()

for field, code in SENTINELS.items():
    is_code = orig_clean[field] == code        # True for the rows that hold the code
    orig_clean.loc[is_code, field] = np.nan     # those rows become missing

**What:** Counts each code **before** (in `orig`) and **after** (in `orig_clean`), and the missing values after.

**Why:** It links every conversion to a page of the guide and shows how many loans it affects. The benchmark's
answer key must be defensible from the documentation.

**Check:** DTI `999`: 4,020 before, 0 after, 4,020 missing. LTV and CLTV `999`: 1 loan each. The other codes do not
occur in this sample (0). Every "after" count is 0.

In [ ]:
table_rows = []
for field, code in SENTINELS.items():
    count_before = int((orig[field] == code).sum())
    count_after = int((orig_clean[field] == code).sum())
    missing_after = int(orig_clean[field].isna().sum())
    table_rows.append({
        "field": field,
        "code": code,
        "meaning": GUIDE_MEANING[field],
        "source": "General User Guide, July 2026, p. " + GUIDE_PAGE[field],
        "code count before": count_before,
        "code count after": count_after,
        "missing after": missing_after,
    })
na_table = pd.DataFrame(table_rows)

# Show the long "meaning" texts in full (not cut off with "...").
with pd.option_context("display.max_colwidth", None):
    display(na_table)

**What:** Counts how often DTI = `999` occurs together with the HARP indicator. HARP (Home Affordable Refinance
Program; Freddie Mac's "Relief Refinance") helped borrowers refinance after home prices fell.

**Why:** The guide says *"All HARP loans in the dataset will be disclosed as Not Available"* for DTI. So a missing
DTI mostly means "this is a HARP refinance", not "the borrower hid their income". That is why the baseline keeps
"DTI was missing" as its own feature, `dti_missing`.

**Check:** 4,017 of the 4,020 loans with DTI 999 are HARP loans, and every HARP loan has DTI 999. The other 3 are
not HARP loans; the guide's other reason for 999 is a ratio above 65%.

In [ ]:
dti_is_999 = orig["orig_dti"] == "999"
dti_is_999 = dti_is_999.rename("DTI is 999")                 # the name shown in the table
harp = orig["harp_indicator"].rename("HARP indicator")

# A count table: rows = "DTI is 999" (False/True), columns = HARP (N/Y). margins=True adds the "All" totals.
display(pd.crosstab(dti_is_999, harp, margins=True))

is_harp = orig["harp_indicator"] == "Y"
n_999 = int(dti_is_999.sum())
n_999_and_harp = int((dti_is_999 & is_harp).sum())
print(f"{n_999_and_harp:,} of {n_999:,} loans with DTI 999 are HARP loans")

**What:** Converts the 10 numeric fields from text to numbers.

**Why:** The model needs numbers. This works now because the codes are gone; a missing value (NaN) stays missing.

**Check:** every field has a number type. Whole numbers are `int64`; fields with decimals or missing values are
`float64` (NaN is a decimal number in pandas, so a column with NaN cannot be `int64`).

In [ ]:
NUMERIC_ORIG_FIELDS = ["fico", "mi_pct", "num_units", "orig_cltv", "orig_dti", "orig_ltv",
                       "orig_interest_rate", "orig_upb", "orig_loan_term", "num_borrowers"]

for field in NUMERIC_ORIG_FIELDS:
    orig_clean[field] = pd.to_numeric(orig_clean[field])

print(orig_clean[NUMERIC_ORIG_FIELDS].dtypes)

**The five derived features, one sentence each.** (A *derived* feature is a new column computed from other columns.)

* `dti_missing` = 1 when DTI was the 999 code (now NaN). The model can then treat "DTI not disclosed" (almost always
  a HARP refinance) separately from the DTI value, which section 8 fills with the training median.
* `log_orig_upb` = natural log of the original loan amount (UPB = unpaid principal balance). A few loans are very
  large; on the log scale, doubling the loan size is always the same step.
* `term_360` = 1 for a 360-month (30-year) loan. Shorter terms, mostly 15-year loans, are paid back faster.
* `multi_borrower` = 1 when two or more borrowers are on the loan. A missing count (code 99) would give 0, but no
  loan in this sample has that code.
* `mi_flag` = 1 when the loan has mortgage insurance (`mi_pct > 0`). A missing value (code 999) would give 0, but
  no loan in this sample has that code.

**What:** Adds the five derived features to `orig_clean`. Most of them are first a True/False column, then turned
into 1/0.

**Why:** These are model inputs (section 8). `astype(int)` turns True into 1 and False into 0.

**Check:** nothing is printed; the next cells show the results.

In [ ]:
# 1. dti_missing: 1 if DTI is missing (it was the 999 code).
dti_is_missing = orig_clean["orig_dti"].isna()
orig_clean["dti_missing"] = dti_is_missing.astype(int)

# 2. log_orig_upb: natural log of the original loan amount.
orig_clean["log_orig_upb"] = np.log(orig_clean["orig_upb"])

# 3. term_360: 1 for a 30-year (360-month) loan.
is_30_year_loan = orig_clean["orig_loan_term"] == 360
orig_clean["term_360"] = is_30_year_loan.astype(int)

# 4. multi_borrower: 1 for two or more borrowers.
has_2_or_more_borrowers = orig_clean["num_borrowers"] >= 2
orig_clean["multi_borrower"] = has_2_or_more_borrowers.astype(int)

# 5. mi_flag: 1 if the loan has mortgage insurance.
has_mortgage_insurance = orig_clean["mi_pct"] > 0
orig_clean["mi_flag"] = has_mortgage_insurance.astype(int)

**What:** Keeps the 16 columns the model uses (the loan ID, 10 numeric and 5 categorical fields) in a new table,
`loan_features`, and summarizes its numeric columns.

**Why:** `describe()` is a quick sanity check: impossible values would point to a wrong column order or a missed
code.

**Check:** shape (50000, 16). FICO between 300 and 850; `orig_dti` has a count of 45,980 (4,020 missing) and a
maximum of at most 65; `orig_ltv` has a count of 49,999 (one missing). LTV values above 105% are possible: the guide
(p. 6) allows 1–998% for HARP loans.

In [ ]:
LOAN_FEATURE_COLUMNS = [
    "loan_id",
    # numeric
    "fico", "orig_ltv", "orig_dti", "dti_missing", "orig_interest_rate",
    "log_orig_upb", "term_360", "multi_borrower", "mi_flag", "num_units",
    # categorical (text codes)
    "loan_purpose", "occupancy_status", "first_time_homebuyer", "property_type", "channel",
]

loan_features = orig_clean[LOAN_FEATURE_COLUMNS]     # one row per loan
print(loan_features.shape)

# describe() gives count, mean, sd, min, quartiles and max of each numeric column.
# .T turns the table so that each column of loan_features becomes one row.
numeric_summary = loan_features.describe()
display(numeric_summary.T.round(2))

**What:** Shows the raw text next to the derived value for 5 loans: the first 4 loans with a reported DTI, and the
first loan with DTI 999.

**Why:** It shows on real rows that each derived feature does what its sentence above says.

**Check:** in the DTI-999 row, `dti_missing` = 1. `term_360` is 1 exactly when the term is `360`; `multi_borrower` is
1 when there are 2 borrowers; `mi_flag` is 1 when `mi_pct` is above 0.

In [ ]:
# Pick the rows: the first 4 loans with a reported DTI, and the first loan with DTI 999.
has_reported_dti = orig["orig_dti"] != "999"
first_4_with_dti = list(orig[has_reported_dti].index[:4])
first_with_999 = orig[~has_reported_dti].index[0]          # ~ means "not"
rows = first_4_with_dti + [first_with_999]

before_after = pd.DataFrame({
    "orig_dti (raw)": orig.loc[rows, "orig_dti"],
    "dti_missing": loan_features.loc[rows, "dti_missing"],
    "orig_upb (raw)": orig.loc[rows, "orig_upb"],
    "log_orig_upb": loan_features.loc[rows, "log_orig_upb"].round(3),
    "orig_loan_term (raw)": orig.loc[rows, "orig_loan_term"],
    "term_360": loan_features.loc[rows, "term_360"],
    "num_borrowers (raw)": orig.loc[rows, "num_borrowers"],
    "multi_borrower": loan_features.loc[rows, "multi_borrower"],
    "mi_pct (raw)": orig.loc[rows, "mi_pct"],
    "mi_flag": loan_features.loc[rows, "mi_flag"],
})
display(before_after)

**What:** Counts the 0s and 1s of the four 0/1 features.

**Why:** A flag that is almost always 0 (or always 1) carries little information.

**Check:** `dti_missing` = 1 for 4,020 loans; `term_360` = 1 for 36,501; `multi_borrower` = 1 for 25,540; `mi_flag` =
1 for 11,729.

In [ ]:
flag_counts = pd.DataFrame()
for flag in ["dti_missing", "term_360", "multi_borrower", "mi_flag"]:
    flag_counts[flag] = loan_features[flag].value_counts().sort_index()   # how many 0s, how many 1s
flag_counts.index.name = "value"
display(flag_counts)

**A note on the categorical "9"/"99" codes.** A *categorical* field holds a code for a group, not a number (for
example `loan_purpose` = `P` for purchase). These fields also have "Not Available" codes: `9` for loan purpose,
occupancy status, first-time homebuyer and channel, and `99` for property type (guide pp. 4–8).

The baseline does **not** convert them: they stay as their own category. This matters little, because only
**8 loans** have such a code, all in `first_time_homebuyer`. The encoder in section 8 puts categories seen fewer
than 20 times in training into one shared "infrequent" column, so these 8 loans end up in
`first_time_homebuyer_infrequent_sklearn`. Exercise 1 in section 13 maps them to an explicit "unknown" category.

**What:** Counts the values of the five categorical fields, and the loans with a `9` or `99` code in any of them.

**Why:** It backs the note above with numbers.

**Check:** only `first_time_homebuyer` has a `9` (8 loans). The last line prints 8.

In [ ]:
CATEGORICAL_ORIG_FIELDS = ["loan_purpose", "occupancy_status", "first_time_homebuyer", "property_type", "channel"]

for field in CATEGORICAL_ORIG_FIELDS:
    field_counts = orig[field].value_counts()
    print(f"{field:22s}", field_counts.to_dict())

# A True/False table: True where a field holds the code 9 or 99.
is_unknown_code = orig[CATEGORICAL_ORIG_FIELDS].isin(["9", "99"])

# True for a loan if at least one of its fields holds such a code. any(axis=1) looks across each row.
loan_has_unknown_code = is_unknown_code.any(axis=1)
print("\nLoans with a '9' or '99' code in any of these fields:", int(loan_has_unknown_code.sum()))

## 4. Cleaning the monthly performance data

**The month counter $t$.** Months are stored as `YYYYMM` numbers. They sort correctly, but you cannot subtract
them: $201601 - 201512 = 89$, although the two months are only 1 month apart. So `ym_index()` turns every month into
a simple counter:

$$ t \;=\; 12 \cdot \left\lfloor \frac{\text{YYYYMM}}{100} \right\rfloor \;+\; (\text{YYYYMM} \bmod 100) \;-\; 1 $$

In words, line by line:

* $\lfloor \text{YYYYMM} / 100 \rfloor$ is the year: divide by 100 and round down ($\lfloor\ \rfloor$ means "round
  down"). $201512 / 100 = 2015.12 \to 2015$.
* $\text{YYYYMM} \bmod 100$ is the month: the remainder after dividing by 100. $201512 \bmod 100 = 12$.
* $t$ = year × 12 + month − 1.

Example: $201512 \to 12 \cdot 2015 + 12 - 1 = 24191$ and $201601 \to 12 \cdot 2016 + 1 - 1 = 24192$: exactly one
apart. So "12 months after $t$" is simply $t + 12$.

**What:** Defines `ym_index()`, the month counter described above. In Python, `//` is whole-number division and `%`
is the remainder.

**Why:** Every time window in this notebook (the 12 months up to $t$, the 12 months after $t$) is computed with it.

**Leakage-test cell.** This cell carries the tag `leakage-test-source`: `tests/test_baseline_no_leakage.py` reads it
and runs it together with `build_snapshot()` (section 5; see section 11). Its code is kept exactly as in the original
script. Keep the tag, and change the code only on purpose, as a reviewed change (section 13).

**Check:** nothing is printed; the next cell tries it.

In [ ]:
# ym_index(): a month written as YYYYMM (for example 201712) -> a simple month counter (24215).
# tests/test_baseline_no_leakage.py runs this cell: keep its tag, and change its code only on purpose (section 11).
def ym_index(s):
    """YYYYMM -> months since year 0 (so t+12 is simple integer arithmetic)."""
    s = pd.Series(s).astype(int)                 # the months as whole numbers
    return (s // 100) * 12 + (s % 100) - 1       # year * 12 + month - 1

**What:** Applies `ym_index()` to four months in a row and compares the steps with plain `YYYYMM` subtraction.

**Why:** An error here would shift every window.

**Check:** the `YYYYMM` steps are 1, 89, 1; the `t` steps are always 1. The last line prints `True`.

In [ ]:
months = pd.Series([201511, 201512, 201601, 201602])
month_counter = ym_index(months)

demo = pd.DataFrame({"YYYYMM": months, "t": month_counter})
demo["step in YYYYMM"] = demo["YYYYMM"].diff()     # diff() = this row minus the row above (plain subtraction)
demo["step in t"] = demo["t"].diff()               # the true number of months between the rows
display(demo)

t_dec_2017 = int(ym_index([201712])[0])
t_dec_2018 = int(ym_index([201812])[0])
print("t for 201712 =", t_dec_2017)
print("t for 201712, plus 12, equals t for 201812:", t_dec_2017 + 12 == t_dec_2018)

**Delinquency status codes** (guide p. 10). The status counts from the due date of the last paid installment:

| code | meaning | `dq` (number) | `sev` (serious) |
|---|---|---|---|
| `00` | current, or less than 30 days late | 0 | False |
| `01` | 30–59 days late | 1 | False |
| `02` | 60–89 days late | 2 | False |
| `03`, `04`, … | 90–119 days, 120–149 days, … (up to 99) | 3, 4, … | **True** |
| `RA` | REO acquisition: the lender has taken the house | NaN (not a number) | **True** |
| `XX` | not available | NaN | False |

We add three columns to the monthly table:

* `t`: the month counter (from `ym_index()`);
* `dq`: the status as a number (`RA` becomes NaN);
* `sev`: True when `dq` ≥ 3 **or** the status is `RA`. It means "90+ days late or REO".

`sev` is used twice in section 5: a loan with `sev` True at $t$ is not at risk, and a loan with `sev` True in any
month $t+1 \dots t+12$ is an event. `sev` looks only at each month's own value, not at the path: a loan can jump from
`00` to `03` in one month (a known false-alarm trap in `CLAUDE.md`). The guide also says that, for months up to
September 2025, `00` can also mean "cycle before the first payment date".

**What:** Defines `add_derived_columns()`, which adds `t`, `dq` and `sev` to a monthly table, and runs it on the real
data. Then it converts the balance and the loan age from text to numbers.

**Why:** The same three columns are needed three times: for the real data (here), the toy data (section 5) and the
changed toy data (section 11). One helper function makes sure all three are built the same way. It changes the table
you give it and returns nothing.

**Check:** the table shows the first 5 months of the first loan, with the three new columns. `t` goes up by 1 from row
to row; `dq` is the status as a number; `sev` is False, because the status is below `03`.

In [ ]:
def add_derived_columns(panel):
    """Add t (month counter), dq (status as a number) and sev (90+ days late or REO) to a monthly table."""
    # t: the month as a simple counter. (.values = only the numbers, so pandas does not match row labels.)
    panel["t"] = ym_index(panel["period"]).values

    # dq: the status code as a number. errors="coerce" turns codes that are not numbers ("RA") into NaN.
    status_as_number = pd.to_numeric(panel["delinq_status"], errors="coerce")
    panel["dq"] = status_as_number

    # sev: 90+ days late (dq >= 3) or REO (status "RA").
    is_90_plus = status_as_number >= 3
    is_reo = panel["delinq_status"] == "RA"
    panel["sev"] = is_90_plus | is_reo            # | means "or"


add_derived_columns(perf)

# Two more fields from text to numbers.
perf["current_upb"] = pd.to_numeric(perf["current_upb"])    # the current balance
perf["loan_age"] = pd.to_numeric(perf["loan_age"])          # the loan age, in months

# The first 5 records, with the columns that matter here (the loan ID is left out).
display(perf[["period", "delinq_status", "t", "dq", "sev"]].head(5))

**What:** Counts the monthly records for each delinquency status code, with the `dq` and `sev` values they got.

**Why:** It checks the code-to-number rule on the real data, and shows how rare serious delinquency is.

**Check:** `00` 3,415,433 records; `01` 23,522; `02` 6,686; `03` 3,756; `RA` 567, with `dq` NaN and `sev` True.
Every code from `03` up has `sev` True. The only code that is not a number is `RA`; there are no `XX` records.
21,525 records have `sev` True.

In [ ]:
# The number of records for each status code (sorted by code).
records_per_code = perf["delinq_status"].value_counts().sort_index()

# The dq and sev value of each code. All records with the same code have the same values, so we take the first.
groups_by_code = perf.groupby("delinq_status")
dq_of_code = groups_by_code["dq"].first()
sev_of_code = groups_by_code["sev"].first()

by_code = pd.DataFrame({"records": records_per_code, "dq": dq_of_code, "sev": sev_of_code})

# Show the common codes; summarize the rest.
SHOWN_CODES = ["00", "01", "02", "03", "04", "05", "RA"]
display(by_code.loc[SHOWN_CODES])

other_codes = by_code.drop(SHOWN_CODES)
print(f"Other codes: {len(other_codes)} values from {other_codes.index.min()} to {other_codes.index.max()}, "
      f"{other_codes['records'].sum():,} records, all with sev = True: {bool(other_codes['sev'].all())}")

is_not_a_number = by_code["dq"].isna()
print("Codes that are not numbers:", list(by_code.index[is_not_a_number]))
print("'XX' records:", int((perf["delinq_status"] == "XX").sum()))
print(f"Records with sev = True: {int(perf['sev'].sum()):,} of {len(perf):,}")

**What:** Counts the zero-balance codes. A zero-balance code says why a loan's balance went to zero and the loan
left the data.

**Why:** These codes decide the censored labels in section 5. If a loan leaves before $t+12$ without a 90+ month,
code `01` (prepaid) gives `censored_prepay`, and any other code gives `censored_other_termination`.

**Check:** `01` (prepaid) dominates with 38,212 loans; the others are rare (`16`: 108, `09`: 55, `02`: 45, `96`: 35,
`15`: 18, `03`: 18). Almost all records have no code, because the loan is still active that month.

In [ ]:
# What each code means (guide p. 11).
ZB_MEANING = {
    "01": "Prepaid or matured (voluntary payoff)",
    "02": "Third party sale",
    "03": "Short sale or charge off",
    "09": "REO disposition",
    "15": "Whole loan sale",
    "16": "Reperforming loan securitization",
    "96": "Confirmed underwriting/major servicing defect prior to credit event",
}

zb_table = perf["zero_balance_code"].value_counts().to_frame(name="records")
zb_table["meaning (guide p. 11)"] = zb_table.index.map(ZB_MEANING)

# The label each code leads to, if the loan leaves before t+12 without a 90+ month (section 5).
LABEL_COLUMN = "label if it happens before t+12 with no 90+ month"
zb_table[LABEL_COLUMN] = "censored_other_termination"     # every code ...
zb_table.loc["01", LABEL_COLUMN] = "censored_prepay"      # ... except 01
display(zb_table)

n_without_code = int(perf["zero_balance_code"].isna().sum())
print(f"Records without a zero-balance code (loan still active): {n_without_code:,}")

## 5. A toy example: how labels and features are made (made-up data)

`build_snapshot(perf, T)` turns the monthly table into **one row per at-risk loan** for one prediction month $T$ (a
December). For $T$ = December 2017:

```
                features: months <= t only               label: months t+1 .. t+12 (HORIZON)
   2017-01 ............................ 2017-12 = t | 2018-01 ............................ 2018-12 | 2019-01 ...
   |<------- 12-month lookback: t-11 .. t ------->| |<----------- outcome window ----------------->|
```

The **lookback** is the 12 months up to and including $t$: the features may use it. The **outcome window** is the
12 months after $t$: only the label may use it.

The rules (`CLAUDE.md`, "Outcome specification"):

* **At risk at $t$** (the loan gets a row): it has a record at $t$, no zero-balance code, and status `00`, `01` or
  `02` at $t$ (`sev` is False). A loan that left before $t$ has no record at $t$, because a loan that has ended gets
  no more monthly records (guide p. 16). So checking month $t$ covers "at or before $t$".
* **Event** (`event`, y = 1): any month in $t+1 \dots t+12$ with `sev` True (status `03` or worse, or `RA`).
* **Non-event** (`non_event`, y = 0): no event, and the loan still has a record at month $t+12$.
* **Censored** (counted, but not used to fit the model): no event, and the loan leaves before $t+12$.
  `censored_prepay` = zero-balance code `01`; `censored_other_termination` = any other code;
  `censored_end_of_data` = the records stop without a code (the window runs past the end of the data).

The order matters: the event check comes first. A loan that goes 90+ and then pays off inside the window is still an
**event** (another false-alarm trap in `CLAUDE.md`: loans can prepay while 90+).

This section has four parts: (1) five made-up loans; (2) `build_snapshot()` itself, read line by line; (3) the same
steps done slowly, by hand, on the made-up loans; (4) a check that (2) and (3) give the same table.

**The five made-up loans.** IDs `X01`–`X05` (on purpose not the Freddie Mac `F15Q…` pattern). Prediction month
$T$ = December 2017, so the lookback is 2017-01 … 2017-12 and the outcome window is 2018-01 … 2018-12.

| loan | story | expected |
|---|---|---|
| `X01` | current the whole time | at risk → `non_event` |
| `X02` | 30 days late in Sep 2017, current at $t$, then 30 → 60 → 90 days late in Mar–May 2018 | at risk → `event` |
| `X03` | 30 days late **at** $t$, current again in Jan 2018, prepays in Apr 2018 | at risk → `censored_prepay` |
| `X04` | 30 → 60 → 90 days late in Oct–Dec 2017: already 90+ **at** $t$ | **not at risk** → no row |
| `X05` | current through Dec 2018; late only from Jan 2019 ($t+13$) | at risk → `non_event` |

**What:** Writes down each made-up loan as one line of monthly status codes, from 201701 to 201906.

**Why:** With five loans and 30 months, every label and feature can be checked by hand. As a timeline, each loan's
story is visible at a glance.

**Check:** read each line as a timeline. The column under the `D` of 2017 is month $t$. `X04` is `03` there; `X03` is
`01` there, and `--` (no record) after its payoff month 201804.

In [ ]:
T_TOY = 201712      # the toy prediction month t (December 2017)

# The 30 toy months: 201701, 201702, ..., 201906.
TOY_PERIODS = []
for year in [2017, 2018, 2019]:
    for month in range(1, 13):
        TOY_PERIODS.append(f"{year}{month:02d}")      # :02d writes the month 1 as "01"
TOY_PERIODS = TOY_PERIODS[:30]                         # keep the first 30 months: stop at 201906

# The monthly status of each made-up loan: one two-character code per month; "--" = no record.
TOY_STATUS = {
    #       2017 (t = December 2017)              |  2018 = outcome window t+1..t+12      |  2019
    #       J  F  M  A  M  J  J  A  S  O  N  D   |  J  F  M  A  M  J  J  A  S  O  N  D   |  J  F  M  A  M  J
    "X01": "00 00 00 00 00 00 00 00 00 00 00 00  |  00 00 00 00 00 00 00 00 00 00 00 00  |  00 00 00 00 00 00",   # stays current
    "X02": "00 00 00 00 00 00 00 00 01 00 00 00  |  00 00 01 02 03 04 05 06 07 08 09 10  |  11 12 13 14 15 16",   # 90+ within 12 months
    "X03": "00 00 00 00 00 00 00 00 00 00 00 01  |  00 00 00 00 -- -- -- -- -- -- -- --  |  -- -- -- -- -- --",   # prepays in 201804
    "X04": "00 00 00 00 00 00 00 00 00 01 02 03  |  04 05 06 07 08 09 10 11 12 13 14 15  |  16 17 18 19 20 21",   # already 90+ at t
    "X05": "00 00 00 00 00 00 00 00 00 00 00 00  |  00 00 00 00 00 00 00 00 00 00 00 00  |  01 02 03 04 05 06",   # late only after t+12
}

**What:** Turns the timelines into a monthly table, `toy`, with the same columns and types as the real `perf`. Then
it adds `t`, `dq` and `sev` with `add_derived_columns()` (section 4).

**Why:** `build_snapshot()` must get the same kind of table as in the real run, so that the toy result means
something for the real data. Balances and loan ages are made up; the modification, disaster and assistance fields
are left empty.

**Check:** 136 rows: 30 months for four loans, and 16 for `X03`.

In [ ]:
TOY_AGE_JAN2017 = {"X01": 25, "X02": 7, "X03": 13, "X04": 19, "X05": 31}                         # loan age in 201701
TOY_UPB = {"X01": 210000.0, "X02": 145000.0, "X03": 320000.0, "X04": 98000.0, "X05": 175000.0}   # balance (flat, made up)

toy_rows = []
for loan_id, timeline in TOY_STATUS.items():
    # 1. Split the timeline into its 30 monthly codes (leave out the "|" separators).
    codes = []
    for piece in timeline.split():
        if piece != "|":
            codes.append(piece)
    assert len(codes) == len(TOY_PERIODS)

    # 2. Find the position of the loan's last record (the last code that is not "--").
    last_position = 0
    for position in range(len(codes)):
        if codes[position] != "--":
            last_position = position

    # 3. A loan whose line ends with "--" leaves the data early: it prepaid on its last record.
    leaves_early = codes[-1] == "--"

    # 4. One row per month that has a record.
    for position in range(len(codes)):
        status = codes[position]
        if status == "--":
            continue                                   # no record: the loan has left the data
        if leaves_early and position == last_position:
            zero_balance_code = "01"                   # prepaid on this record
        else:
            zero_balance_code = None                   # still active
        toy_rows.append({
            "loan_id": loan_id,
            "period": TOY_PERIODS[position],
            "current_upb": TOY_UPB[loan_id],
            "delinq_status": status,
            "loan_age": TOY_AGE_JAN2017[loan_id] + position,
            "modification_flag": None,
            "zero_balance_code": zero_balance_code,
            "delinq_due_to_disaster": None,
            "borrower_assistance_plan": None,
        })

toy = pd.DataFrame(toy_rows)

# The text columns get the same type as when the real files are read (None becomes a missing value).
TEXT_COLUMNS = ["loan_id", "period", "delinq_status", "modification_flag", "zero_balance_code",
                "delinq_due_to_disaster", "borrower_assistance_plan"]
for column in TEXT_COLUMNS:
    toy[column] = toy[column].astype("str")

add_derived_columns(toy)                               # t, dq and sev, exactly as for the real data

print("Toy table:", toy.shape)
print("Records per loan:", toy["loan_id"].value_counts().sort_index().to_dict())

**What:** Shows the `X03` records from 201710 to 201805.

**Why:** This is what one loan looks like in the long, one-row-per-month table that `build_snapshot()` receives.

**Check:** 201712 (= $t$) has status `01`; 201804 is the last record and has zero-balance code `01`; there is no
201805 record.

In [ ]:
is_x03 = toy["loan_id"] == "X03"
in_shown_months = toy["period"].between("201710", "201805")
display(toy[is_x03 & in_shown_months])

### The function `build_snapshot()`

**What:** Defines `build_snapshot(perf, T)`. It has three blocks: (1) who is at risk at $t$; (2) the features,
computed **only** from records at or before $t$; (3) the label, from the months $t+1 \dots t+$`HORIZON`.

**Why:** This is the only place where time enters the pipeline. A mistake here means leakage (features that see the
future) or a wrong outcome. Benchmark pairs 3 (horizon) and 4 (leakage) plant their defects exactly here.

**Leakage-test cell.** Like `ym_index()`, this cell carries the tag `leakage-test-source`, and
`tests/test_baseline_no_leakage.py` runs it (section 11). The test also edits one line of it on purpose (the line
that builds `h12`) to check that it can catch a leak. So the code is kept exactly as in the original script, short
names included; change it only on purpose, as a reviewed change (section 13). Instead of rewriting the code, this
notebook explains it: the numbers `[1]` … `[23]` in the comments point to the walkthrough below, and the slow version
after it does every step by hand.

**The short names inside the function:**

| name | meaning |
|---|---|
| `t` | the prediction month $T$ as a month counter |
| `hist` | "history": every record at or before $t$ |
| `now` | the records **at** month $t$ (after step [4]: only the at-risk loans) |
| `ids` | the loan IDs of the at-risk loans |
| `h12` | the last 12 months of history, $t-11 \dots t$, of the at-risk loans |
| `g12` | `h12` grouped by loan (to get one number per loan) |
| `ever` | the whole history of the at-risk loans, grouped by loan |
| `hh` | the whole history of the at-risk loans, not grouped |
| `feat` | the feature table: one row per at-risk loan |
| `win` | the outcome window $t+1 \dots t+$`HORIZON` (used **only** for the label) |
| `event`, `last_t`, `zb` | per loan: any 90+ month in the window? last month seen? zero-balance code? |

**Check:** nothing is printed. Every feature line uses `hist`, `now`, `h12`, `g12`, `ever` or `hh`, which all hold
only months ≤ $t$. Only the label block (Block 3) uses `perf` directly.

In [ ]:
# build_snapshot(perf, T): one row per at-risk loan for the prediction month T (a December, written YYYYMM),
# with its features (from months <= t only) and its label (from months t+1 .. t+HORIZON).
# tests/test_baseline_no_leakage.py runs this cell, and edits the h12 line on purpose to check that it catches a
# leak: keep the tag and the h12 line, and change the code only on purpose (section 11).
# The numbers [1] .. [23] point to the walkthrough below.
def build_snapshot(perf, T):
    # ---- Block 1: who is at risk at month t?
    t = int(ym_index([T])[0])                                  # [1] prediction month as a month index
    hist = perf[perf.t <= t]                                   # [2] ONLY information at or before t
    now = hist[hist.t == t]                                    # [3] each loan's record AT month t
    now = now[now.zero_balance_code.isna() & ~now.sev]         # [4] at-risk population
    ids = now.loan_id                                          # [5] the at-risk loan IDs

    # ---- Block 2: features, computed only from records at or before t
    h12 = hist[(hist.t > t - 12) & hist.loan_id.isin(ids)]     # [6] lookback window t-11 .. t
    g12 = h12.groupby("loan_id")                               # [7] the lookback, grouped by loan
    ever = hist[hist.loan_id.isin(ids)].groupby("loan_id")     # [8] the whole history up to t, grouped by loan
    feat = now.set_index("loan_id")[["delinq_status", "loan_age", "current_upb",
                                     "delinq_due_to_disaster", "borrower_assistance_plan"]].copy()   # [9]
    feat["status_t"] = feat.pop("delinq_status")               # [10] '00', '01', '02'
    feat["max_dq_12m"] = g12.dq.max().clip(upper=3)            # [11] 0..3 (3 = was 90+ in last 12m)
    feat["n_dq_months_12m"] = (h12.dq >= 1).groupby(h12.loan_id).sum()        # [12] months 30+ days late in t-11..t
    feat["ever_90_before_t"] = ever.sev.any().astype(int)      # [13] ever 90+ or REO at or before t
    hh = hist[hist.loan_id.isin(ids)]                          # [14] the whole history up to t, not grouped
    feat["ever_modified"] = hh.modification_flag.notna().groupby(hh.loan_id).any().astype(int)   # [15] any Y/P flag
    feat["disaster_flag_t"] = feat.pop("delinq_due_to_disaster").eq("Y").astype(int)  # [16] disaster flag at t
    feat["assist_plan_t"] = feat.pop("borrower_assistance_plan").fillna("none")       # [17] F/R/T at t, else 'none'
    # Leakage is tested by tests/test_baseline_no_leakage.py (future-invariance + mutation check).

    # ---- Block 3: label (future window, never used as a feature)
    win = perf[(perf.t > t) & (perf.t <= t + HORIZON) & perf.loan_id.isin(ids)]      # [18] months t+1 .. t+HORIZON
    event = win.groupby("loan_id").sev.any().reindex(ids, fill_value=False)          # [19] any 90+/REO month?
    last_t = perf[perf.loan_id.isin(ids)].groupby("loan_id").t.max().reindex(ids)     # [20] last month observed
    zb = (perf[perf.loan_id.isin(ids) & perf.zero_balance_code.notna()]
          .set_index("loan_id").zero_balance_code.reindex(ids))                        # [21] zero-balance code, if any
    label = np.where(event.values, "event",                                            # [22] event?
             np.where(last_t.values >= t + HORIZON, "non_event",                       #      still seen at t+HORIZON?
              np.where(zb.values == "01", "censored_prepay",                           #      prepaid?
               np.where(pd.notna(zb.values), "censored_other_termination", "censored_end_of_data"))))   # other code?
    feat["label"] = label                                      # [23] add the label ...
    feat["snapshot"] = T                                       #      ... and the snapshot month,
    return feat.reset_index()                                  #      and make loan_id a normal column again

### `build_snapshot()` line by line

The numbers match the `[1]` … `[23]` comments in the code. The toy values are for $T$ = 201712.

**Block 1: who is at risk at month $t$?**

1. `t = int(ym_index([T])[0])`: turn the December into the month counter. Toy: 201712 → 24215.
2. `hist = perf[perf.t <= t]`: keep only the past, the records at or before $t$. All features come from `hist`.
   Toy: the months 201701–201712.
3. `now = hist[hist.t == t]`: the record of each loan **in** month $t$. Toy: 5 records, one per loan.
4. `now = now[now.zero_balance_code.isna() & ~now.sev]`: keep the loans that have no zero-balance code **and**
   (`&`) are **not** (`~`) 90+/REO. Toy: `X04` is dropped (status `03`).
5. `ids = now.loan_id`: the IDs of the at-risk loans. Toy: `X01`, `X02`, `X03`, `X05`.

**Block 2: the features (only months ≤ $t$).**

6. `h12 = hist[(hist.t > t - 12) & hist.loan_id.isin(ids)]`: the lookback. `hist` already stops at $t$, and
   `hist.t > t - 12` keeps $t-11 \dots t$. `isin(ids)` keeps only the at-risk loans. This is the line that the
   leakage test edits on purpose (section 11).
7. `g12 = h12.groupby("loan_id")`: group the lookback rows by loan, to compute one number per loan.
8. `ever = hist[hist.loan_id.isin(ids)].groupby("loan_id")`: the whole history up to $t$, grouped by loan.
9. `feat = now.set_index("loan_id")[[...]].copy()`: start the feature table. One row per loan (the loan ID becomes
   the row label), with five fields copied from the month-$t$ record: status, loan age, current balance, disaster
   flag and assistance plan.
10. `feat["status_t"] = feat.pop("delinq_status")`: rename `delinq_status` to `status_t`. (`pop` takes a column out of
    the table and gives it back, so it can be stored under the new name.) Toy: `X03` → `01`.
11. `feat["max_dq_12m"] = g12.dq.max().clip(upper=3)`: the worst status in the lookback, capped at 3
    (`clip(upper=3)` turns anything above 3 into 3). Toy: `X02` → 1 (30 days late in 201709).
12. `feat["n_dq_months_12m"] = (h12.dq >= 1).groupby(h12.loan_id).sum()`: the number of lookback months that were at
    least 30 days late. (`dq >= 1` gives True/False; the sum counts the Trues.) Toy: `X02` → 1.
13. `feat["ever_90_before_t"] = ever.sev.any().astype(int)`: 1 if the loan was ever 90+/REO at or before $t$. Toy: 0
    for all.
14. `hh = hist[hist.loan_id.isin(ids)]`: the whole history up to $t$ again, this time not grouped.
15. `feat["ever_modified"] = hh.modification_flag.notna().groupby(hh.loan_id).any().astype(int)`: 1 if any record up
    to $t$ has a modification flag (`Y` or `P`). Toy: 0 for all.
16. `feat["disaster_flag_t"] = feat.pop("delinq_due_to_disaster").eq("Y").astype(int)`: 1 if the disaster flag at $t$
    is `Y` (`.eq("Y")` means `== "Y"`). Toy: 0 for all.
17. `feat["assist_plan_t"] = feat.pop("borrower_assistance_plan").fillna("none")`: the assistance plan at $t$ (`F`,
    `R` or `T`), or `none` if the field is empty. Toy: `none` for all.

**Block 3: the label (the only block that looks after $t$).**

18. `win = perf[(perf.t > t) & (perf.t <= t + HORIZON) & perf.loan_id.isin(ids)]`: the outcome window $t+1 \dots
    t+12$ of the at-risk loans. It uses `perf`, not `hist`, because the label **must** look at the future.
19. `event = win.groupby("loan_id").sev.any().reindex(ids, fill_value=False)`: True if any window month is 90+/REO.
    `reindex(ids, fill_value=False)` gives every at-risk loan a value, in the order of `ids`, and False for a loan
    with no window records at all. Toy: `X02` → True.
20. `last_t = perf[perf.loan_id.isin(ids)].groupby("loan_id").t.max().reindex(ids)`: the last month in which we see
    each loan. Toy: `X03` → the counter of 201804.
21. `zb = (...).zero_balance_code.reindex(ids)`: each loan's zero-balance code, or missing if it has none. Toy:
    `X03` → `01`.
22. `label = np.where(...)`: the decision. `np.where(condition, A, B)` means "A where the condition is True, B where
    it is False". Four of them, one inside the other, give five outcomes, for all loans at once:

    ```
    if   the loan has a 90+/REO month in the window:     label = "event"
    elif we still see the loan at t + HORIZON:           label = "non_event"
    elif its zero-balance code is "01" (prepaid):        label = "censored_prepay"
    elif it has another zero-balance code:               label = "censored_other_termination"
    else (its records just stop):                        label = "censored_end_of_data"
    ```

23. The last three lines add the label and the snapshot month, and turn the loan ID from the row label back into a
    normal column (`reset_index()`).

### The same steps, written out slowly (toy data)

The next four cells repeat `build_snapshot()` by hand on the toy loans, with long names and one small step per line.
The features and the label use a plain loop over the four at-risk loans. That is fine for 4 loans; on the 3.5
million real records a loop would be far too slow, which is why the real function works on all loans at once. The
last cell checks that the slow version and `build_snapshot()` give exactly the same values.

**What:** Block 1, slowly: the month counter, the history, the records at $t$, and the at-risk test.

**Why:** It shows which loans get a row at all, before any feature or label is computed.

**Check:** `t = 24215`. `X04` has `sev` True at $t$ (status `03`), so `at_risk` is False. The other four loans are at
risk, including `X03` with status `01`.

In [ ]:
t = int(ym_index([T_TOY])[0])                              # [1]
print("t =", t)

history = toy[toy["t"] <= t]                               # [2] hist: every record at or before t
records_at_t = history[history["t"] == t]                  # [3] the record of each loan in month t

no_zero_balance_code = records_at_t["zero_balance_code"].isna()
not_90_plus = ~records_at_t["sev"]                         # ~ means "not"
is_at_risk = no_zero_balance_code & not_90_plus            # [4] & means "and"

at_risk_ids = list(records_at_t.loc[is_at_risk, "loan_id"])     # [5] ids
print("At-risk loans:", at_risk_ids)

at_risk_check = records_at_t[["loan_id", "period", "delinq_status", "zero_balance_code", "sev"]].copy()
at_risk_check["at_risk"] = is_at_risk
display(at_risk_check)

**What:** Block 2, slowly: the features, loan by loan, from the records at or before $t$ only.

**Why:** Each feature is one plain step here, so you can compare it with the line of `build_snapshot()` it repeats
(the number in the comment).

**Check:** `X02` and `X03` have `max_dq_12m` = 1 and `n_dq_months_12m` = 1; `X03` has `status_t` = `01`; every other
feature is 0 or `none`.

In [ ]:
feature_rows = []
for loan_id in at_risk_ids:
    # This loan's records at or before t, its lookback months t-11 .. t, and its record at t.
    loan_history = history[history["loan_id"] == loan_id]
    loan_lookback = loan_history[loan_history["t"] > t - 12]                 # [6]
    record_at_t = loan_history[loan_history["t"] == t].iloc[0]                # .iloc[0]: the one row, as a Series

    # [11] The worst status in the lookback, capped at 3.
    worst_status = loan_lookback["dq"].max()
    max_dq_12m = min(worst_status, 3)

    # [12] The number of lookback months that were at least 30 days late.
    is_late_month = loan_lookback["dq"] >= 1
    n_dq_months_12m = int(is_late_month.sum())

    # [13] Ever 90+ or REO at or before t?
    ever_90_before_t = int(loan_history["sev"].any())

    # [15] Any modification flag at or before t?
    has_modification_flag = loan_history["modification_flag"].notna()
    ever_modified = int(has_modification_flag.any())

    # [16] Disaster flag at t?
    disaster_flag_t = int(record_at_t["delinq_due_to_disaster"] == "Y")

    # [17] The assistance plan at t, or "none".
    plan_at_t = record_at_t["borrower_assistance_plan"]
    if pd.isna(plan_at_t):
        assist_plan_t = "none"
    else:
        assist_plan_t = plan_at_t

    # [9], [10]: the fields copied from the record at t, and all the features of this loan.
    feature_rows.append({
        "loan_id": loan_id,
        "loan_age": record_at_t["loan_age"],
        "current_upb": record_at_t["current_upb"],
        "status_t": record_at_t["delinq_status"],
        "max_dq_12m": max_dq_12m,
        "n_dq_months_12m": n_dq_months_12m,
        "ever_90_before_t": ever_90_before_t,
        "ever_modified": ever_modified,
        "disaster_flag_t": disaster_flag_t,
        "assist_plan_t": assist_plan_t,
    })

slow_features = pd.DataFrame(feature_rows)
display(slow_features)

**What:** Block 3, slowly: the label, loan by loan, from the outcome window $t+1 \dots t+12$.

**Why:** The label is a small decision (event? → still seen at $t+12$? → which code?). Seeing its inputs makes every
label explainable.

**Check:** `X02`: event True, first 90+ month 201805. `X03`: only 4 window months, last seen 201804, code `01` →
`censored_prepay`. `X01` and `X05`: no 90+ month, seen at 201812 → `non_event`.

In [ ]:
label_rows = []
for loan_id in at_risk_ids:
    # Every record of this loan, also after t: the label is allowed to look at the future.
    loan_records = toy[toy["loan_id"] == loan_id]

    # [18] The outcome window t+1 .. t+HORIZON.
    is_after_t = loan_records["t"] > t
    is_before_horizon_end = loan_records["t"] <= t + HORIZON
    loan_window = loan_records[is_after_t & is_before_horizon_end]

    # [19] Any 90+/REO month in the window?
    has_event = bool(loan_window["sev"].any())

    # [20] The last month in which we see the loan. Do we still see it at t + HORIZON?
    last_t_seen = loan_records["t"].max()
    seen_at_horizon_end = bool(last_t_seen >= t + HORIZON)

    # [21] The loan's zero-balance code (None if it has none).
    loan_codes = loan_records["zero_balance_code"].dropna()
    if len(loan_codes) > 0:
        loan_zb_code = loan_codes.iloc[0]
    else:
        loan_zb_code = None

    # [22] The decision, in the same order as in build_snapshot().
    if has_event:
        label = "event"
    elif seen_at_horizon_end:
        label = "non_event"
    elif loan_zb_code == "01":
        label = "censored_prepay"
    elif loan_zb_code is not None:
        label = "censored_other_termination"
    else:
        label = "censored_end_of_data"

    # Only for the display: the first 90+ month in the window (None if there is none).
    severe_window_months = loan_window[loan_window["sev"]]
    if len(severe_window_months) > 0:
        first_90_month = severe_window_months["period"].min()
    else:
        first_90_month = None

    label_rows.append({
        "loan_id": loan_id,
        "months in window": len(loan_window),
        "any 90+ in window (event)": has_event,
        "first 90+ month": first_90_month,
        "last month seen": loan_records["period"].max(),
        f"seen at t+{HORIZON}": seen_at_horizon_end,
        "zero-balance code": loan_zb_code,
        "label": label,
    })

slow_labels = pd.DataFrame(label_rows)
display(slow_labels)

**What:** Runs the real `build_snapshot()` on the toy table, and checks column by column that it gives the same
values as the slow version.

**Why:** This is the format of every real snapshot in section 6: one row per at-risk loan, with its features, its
label and the snapshot month. The check shows that the walkthrough above describes what the function really does.

**Check:** four rows, `X01`, `X02`, `X03`, `X05` (no `X04`), with labels `non_event`, `event`, `censored_prepay`,
`non_event`. Every column says `True`, and the last line says `The slow version and build_snapshot() agree: True`.

In [ ]:
snap_toy = build_snapshot(toy, T_TOY)
display(snap_toy)

# Put the slow results together, in the same column order as build_snapshot().
slow_snapshot = slow_features.copy()
slow_snapshot["label"] = slow_labels["label"]
slow_snapshot["snapshot"] = T_TOY

# Compare column by column: the same values, in the same rows?
all_columns_same = True
for column in snap_toy.columns:
    same = list(snap_toy[column]) == list(slow_snapshot[column])
    print(f"{column:18s} same: {same}")
    if not same:
        all_columns_same = False
print("\nThe slow version and build_snapshot() agree:", all_columns_same)

### Walking through the toy snapshot, loan by loan

**What the columns mean.** `loan_age` and `current_upb` are copied from the month-$t$ record (`current_upb` is in the
snapshot but is **not** a model feature; see exercise 2). `status_t` is the status at $t$: always `00`, `01` or
`02`, because only at-risk loans are here. `max_dq_12m` is the worst status in $t-11 \dots t$, capped at 3.
`n_dq_months_12m` counts the months in $t-11 \dots t$ that were at least 30 days late. `ever_90_before_t` = 1 if the
loan was ever 90+ or REO at or before $t$; since loans that are 90+ **at** $t$ are left out, this can only be a loan
that was 90+ earlier and then recovered. `ever_modified` = 1 if any record up to $t$ has a modification flag (`Y` or
`P`). `disaster_flag_t` = 1 if the disaster flag is `Y` at $t$. `assist_plan_t` is the borrower-assistance plan at
$t$ (`F` forbearance, `R` repayment, `T` trial) or `none`. The toy loans have no modification, disaster or
assistance records, so those three are 0 / `none` for all of them; the made-up loans `T06`–`T08` in
`tests/test_baseline_no_leakage.py` test them.

**`X01`, stays current.** It has a record at $t$, no zero-balance code, and status `00`, so it is at risk. Every
lookback month is `00`: `max_dq_12m` = 0, `n_dq_months_12m` = 0, `ever_90_before_t` = 0. No window month is 90+, and
the loan still has a record in 201812 ($t+12$), so the label is **`non_event`** (y = 0). `loan_age` = 25 + 11 = 36.

**`X02`, becomes 90+ within 12 months.** Status `00` at $t$, so it is at risk. Its 30-day delinquency in 201709 lies
inside the lookback (201701–201712), so `max_dq_12m` = 1 and `n_dq_months_12m` = 1: the features carry an early
warning sign. In the window it goes `01` (201803), `02` (201804), `03` (201805). The first 90+ month is 201805 =
$t+5$, so the label is **`event`** (y = 1). None of the months after $t$ appears in any feature. `loan_age` = 18.

**`X03`, prepays early.** Status `01` (30 days late) at $t$. It is still at risk: the rule leaves out only `03` and
worse. Month $t$ itself is part of the lookback, so `max_dq_12m` = 1 and `n_dq_months_12m` = 1. The loan is current
again in 201801 and pays off in 201804 (zero-balance code `01`); after that its records stop. It had no 90+ month
before leaving, and it was not seen at $t+12$, so the label is **`censored_prepay`**. Nobody knows whether it would
have gone 90+ had it stayed, so this row is counted but not used to fit the model (section 7). `loan_age` = 24.

**`X04`, already 90+ at $t$.** Status `03` at $t$ means `sev` is True, so it is not at risk and gets **no row**.
Asking "will it become 90+?" makes no sense for a loan that already is.

**`X05`, late only after $t+12$.** Current at $t$ and through the whole window 201801–201812, and seen at 201812, so
the label is **`non_event`**. Its 30-day delinquency in 201901 ($t+13$) and its 90+ month 201903 ($t+15$) fall
outside the 12-month window and do not count. With `HORIZON = 15` it would be an event. `loan_age` = 42.

The two labels the toy does not show: `censored_other_termination` (the loan leaves before $t+12$ with a code other
than `01`, for example `09` REO disposition, without a 90+ month first) and `censored_end_of_data` (the records stop
without any code; section 11 produces it by deleting the future).

## 6. Building the real snapshots

The same function now runs on the real monthly table, once for each December in the settings, in the order of the
original script (train, stress, test). Then the loan features from section 3 are added to every row, matched by
`loan_id`.

**What:** Builds one snapshot per December, stacks them into one table, and adds the loan features
(`loan_features`, section 3) with a merge (a merge joins two tables on a shared column, here `loan_id`).

**Why:** This is the modelling table. A loan appears once per December in which it is at risk, so the same loan can
appear in up to 10 snapshots. `validate="many_to_one"` makes pandas check that each loan has exactly one row in
`loan_features`.

**Check:** one line per December with its number of at-risk loans, then the shape (273985, 27): 273,985 rows, 12
columns from `build_snapshot()` + 15 loan features. It takes several seconds.

In [ ]:
ALL_SNAPSHOTS = TRAIN_SNAPSHOTS + STRESS_SNAPSHOTS + TEST_SNAPSHOTS     # the order of the original script

snapshot_tables = []
for T in ALL_SNAPSHOTS:
    one_snapshot = build_snapshot(perf, T)
    print(f"{T}: {len(one_snapshot):,} at-risk loans")
    snapshot_tables.append(one_snapshot)

# Stack the 10 tables on top of each other.
snapshots = pd.concat(snapshot_tables)

# Add the loan features: every row gets the origination features of its own loan.
snapshots = snapshots.merge(loan_features, on="loan_id", how="left", validate="many_to_one")
print(snapshots.shape)

display(snapshots.head(5))

**What:** Counts the labels per December (all loans, before the train/test split). The reference copy of this table
is `reference/label_counts.csv`.

**Why:** Label counts are the first thing to compare with the reference: if they differ, the outcome rules differ.

**Check:** 10 rows (201512 … 202412) and 4 label columns. December 2019 has 1,199 events, far more than any other
year. There is no `censored_end_of_data` column: every window ends before the data end.

In [ ]:
# A count table: one row per December, one column per label.
label_counts = pd.crosstab(snapshots["snapshot"], snapshots["label"])
display(label_counts)

**What:** Compares the label counts with the reference file `reference/label_counts.csv`.

**Why:** It is an early check of the outcome rules, before any model is fitted. Section 12 repeats it in detail.

**Check:** three times `True`. If not, section 12 lists exactly which counts differ.

In [ ]:
ref_label_counts = pd.read_csv("reference/label_counts.csv", index_col="snapshot")

same_decembers = list(label_counts.index) == list(ref_label_counts.index)
same_labels = list(label_counts.columns) == list(ref_label_counts.columns)

# The numbers can only be compared if the rows and columns are the same.
if same_decembers and same_labels:
    same_numbers = bool((label_counts.to_numpy() == ref_label_counts.to_numpy()).all())
else:
    same_numbers = False

print("Same Decembers:", same_decembers)
print("Same labels:", same_labels)
print("Label counts identical to reference/label_counts.csv:", same_numbers)

**What:** For each December: how many loans are at risk, the event rate among the uncensored loans, and the share
of censored loans.

**Why:** The event rate is the base rate the model has to learn. How it changes over time explains the stress-test
design and the calibration gap in section 9.

**Check:** training years about 0.24–0.65%; 201912 about 4.9%; test years about 0.67–0.90%, higher than in training.
The censored share (mostly prepayments) is highest in 201912 and 202012.

In [ ]:
# Which set each December belongs to.
used_as = {}
for T in TRAIN_SNAPSHOTS:
    used_as[T] = "train"
for T in STRESS_SNAPSHOTS:
    used_as[T] = "stress"
for T in TEST_SNAPSHOTS:
    used_as[T] = "test"

at_risk_loans = label_counts.sum(axis=1)                  # all labels together (sum across each row)
events = label_counts["event"]
non_events = label_counts["non_event"]
censored = at_risk_loans - events - non_events            # every other label is a censored label

rates = pd.DataFrame(index=label_counts.index)
rates["used as"] = pd.Series(used_as)
rates["at-risk loans"] = at_risk_loans
rates["events"] = events
rates["event rate % (uncensored)"] = (100 * events / (events + non_events)).round(2)
rates["censored %"] = (100 * censored / at_risk_loans).round(1)
display(rates)

**The December 2019 spike.** The window of the December 2019 snapshot is January–December 2020, the first year of
the COVID-19 pandemic. Under the CARES Act (March 2020), many borrowers paused their payments (**forbearance**; the
`borrower_assistance_plan` field shows `F`). The delinquency status counts from the due date of the last paid
installment (guide p. 10), so paused payments age into 30, 60 and 90 days past due, and these loans count as events
under our rule. The next two cells check this on the data: almost all December 2019 events had a forbearance flag
in their window, and their first 90+ month clusters in mid-2020.

This is a different mechanism from the ordinary credit problems the model is meant to learn. That is why December
2019 and December 2020 are used **only as a stress test** (reported, never used to fit or to test), and why the
training windows end by December 2019.

**What:** Defines two small helpers, then counts, for the events of December 2018, 2019 and 2020, how many had a
forbearance flag (`F`) or a disaster flag (`Y`) in their outcome window.

**Why:** It tests the forbearance explanation of the spike with data, instead of only claiming it.

**Check:** 201912: 1,120 of the 1,199 events had a forbearance flag. 201812 (before the pandemic): only 16 of 160.

In [ ]:
def event_loan_ids(T):
    """The IDs of the loans with the label 'event' in the snapshot of December T."""
    is_this_december = snapshots["snapshot"] == T
    is_event = snapshots["label"] == "event"
    return snapshots.loc[is_this_december & is_event, "loan_id"]


def outcome_window_records(T, loan_ids):
    """The monthly records of these loans in the outcome window t+1 .. t+HORIZON of December T."""
    t_of_T = int(ym_index([T])[0])
    is_after_t = perf["t"] > t_of_T
    is_before_horizon_end = perf["t"] <= t_of_T + HORIZON
    is_chosen_loan = perf["loan_id"].isin(loan_ids)
    return perf[is_after_t & is_before_horizon_end & is_chosen_loan]


forbearance_rows = []
for T in [201812, 201912, 202012]:          # the last training December and the two stress Decembers
    event_ids = event_loan_ids(T)
    window = outcome_window_records(T, event_ids)

    in_forbearance = window["borrower_assistance_plan"] == "F"
    has_disaster_flag = window["delinq_due_to_disaster"] == "Y"

    forbearance_rows.append({
        "snapshot": T,
        "events": len(event_ids),
        "with forbearance (F) in window": window.loc[in_forbearance, "loan_id"].nunique(),
        "with disaster flag (Y) in window": window.loc[has_disaster_flag, "loan_id"].nunique(),
    })

forbearance_table = pd.DataFrame(forbearance_rows)
display(forbearance_table.set_index("snapshot"))

**What:** For the December 2019 events, finds the first month in which each loan was 90+.

**Why:** If forbearance drives the spike, the first 90+ months should bunch up a few months after the start of the
pandemic, instead of spreading evenly over the year.

**Check:** few loans in January–April 2020, then a peak in June 2020 (530 loans) and July 2020 (291).

In [ ]:
event_ids_2019 = event_loan_ids(201912)
window_2019 = outcome_window_records(201912, event_ids_2019)

# Keep only the 90+/REO months, then take the earliest one of each loan.
severe_records = window_2019[window_2019["sev"]]
first_90_month = severe_records.groupby("loan_id")["period"].min()

# How many loans had their first 90+ month in each month of 2020.
loans_per_month = first_90_month.value_counts().sort_index()
loans_per_month.name = "Dec-2019 events: loans first 90+ in month"
display(loans_per_month.to_frame().T)          # .T shows the months side by side

## 7. The train/test split

The split is made in **two** ways at once.

**By loan (hash split).** The same loan appears in several Decembers. If a loan were in both training and test, the
test would partly reward the model for remembering that particular loan. So each loan is put in one pool, once and
for good:

$$ \text{bucket} = \text{SHA-256}(\text{loan ID}) \bmod 100, \qquad
   \text{train pool} \iff \text{bucket} < 70 $$

In words: compute the SHA-256 hash of the loan ID (a fixed recipe that turns any text into a long, random-looking
number), take the remainder after dividing by 100 (a "bucket" from 0 to 99), and put the loan in the train pool when
the bucket is below 70 (`TRAIN_LOAN_SHARE`). The same ID always gives the same hash, on any computer, in any run. So
the split is **reproducible without any random seed**, and adding or removing other loans never moves a loan to the
other pool. Because the hash looks random, about 70% of the loans land in the train pool.

**By time.** Training uses the Decembers 2015–2018; testing uses the Decembers 2021–2024. The model is judged on
years it has never seen, as in real use (an *out-of-time* test). The pandemic windows (2019–2020) are left out of
both and only reported as a stress test (section 6).

**What:** Defines `hash_bucket()` and `loan_is_in_train_pool()`: the hash split described above, one step per line.

**Why:** These two small functions decide which loans may ever be used for training.

**Check:** nothing is printed.

In [ ]:
def hash_bucket(loan_id):
    """The loan's bucket, a number from 0 to 99, computed from the SHA-256 hash of its ID."""
    id_as_bytes = loan_id.encode()                           # the hash works on bytes, not on text
    hash_as_hex = hashlib.sha256(id_as_bytes).hexdigest()    # 64 hexadecimal characters (0-9, a-f)
    hash_as_number = int(hash_as_hex, 16)                    # read them as one (very large) number
    return hash_as_number % 100                              # the remainder after dividing by 100


def loan_is_in_train_pool(loan_id):
    """True if the loan may be used for training: its bucket is below TRAIN_LOAN_SHARE (70)."""
    return hash_bucket(loan_id) < TRAIN_LOAN_SHARE

**What:** Shows the hash, the bucket and the pool of the five made-up IDs.

**Why:** It makes the hash split concrete. Run the cell twice, or on another computer: the numbers never change.

**Check:** each ID gets a bucket from 0 to 99; the pool is `True` exactly when the bucket is below 70.

In [ ]:
for loan_id in ["X01", "X02", "X03", "X04", "X05"]:
    hash_as_hex = hashlib.sha256(loan_id.encode()).hexdigest()
    bucket = hash_bucket(loan_id)
    in_pool = loan_is_in_train_pool(loan_id)
    print(f"{loan_id}: sha256 = {hash_as_hex[:12]}...  bucket = {bucket:2d}  ->  train pool: {in_pool}")

**What:** Adds the column `in_train_pool` to the snapshots, and prints the share of loans in the train pool.

**Why:** It checks that the hash split behaves as intended on the real loan IDs.

**Check:** about 70% of the distinct loans are in the train pool.

In [ ]:
# Apply the function to the loan ID of every row: True = train pool.
snapshots["in_train_pool"] = snapshots["loan_id"].map(loan_is_in_train_pool)

# The share of distinct loans (each loan counted once) in the train pool.
one_row_per_loan = snapshots.drop_duplicates("loan_id")
share_in_pool = one_row_per_loan["in_train_pool"].mean()
print(f"{share_in_pool:.1%} of the {len(one_row_per_loan):,} distinct loans in the snapshots are in the train pool")

**Why censored rows are dropped before fitting.** A censored loan left the data (mostly by prepaying) before $t+12$
without a 90+ month. Its outcome is **unknown**: it might have gone 90+ later in the window, had it stayed. Counting
it as y = 0 would assume it would have stayed current; counting it as y = 1 would be wrong too. So the draft outcome
rules leave these rows out of fitting and evaluation, but count and report them (section 6). See
[CLAUDE.md → "Outcome specification"](../CLAUDE.md). Treating prepayment as y = 0 is a sensitivity analysis that is
still waiting for Dr. Ku's decision (exercise 4).

**What:** Keeps only the uncensored rows, creates the target `y`, splits the rows into train / test / stress, and
checks the two split rules with `assert`.

**Why:** These two checks are the guarantees behind every reported number: no loan is in both train and test, and
every training December comes before every test December.

**Check:** `Train rows 101,843 (events 462) | Test rows 15,716 (events 119) | loan overlap = 0`.

In [ ]:
# 1. Keep only the rows with a known outcome (drop the censored rows).
is_uncensored = snapshots["label"].isin(["event", "non_event"])
uncensored = snapshots[is_uncensored].copy()

# 2. The target y: 1 = event, 0 = non_event.
is_event = uncensored["label"] == "event"
uncensored["y"] = is_event.astype(int)

# 3. Split by time (which December) and by loan (train pool or not).
in_train_years = uncensored["snapshot"].isin(TRAIN_SNAPSHOTS)
in_test_years = uncensored["snapshot"].isin(TEST_SNAPSHOTS)
in_stress_years = uncensored["snapshot"].isin(STRESS_SNAPSHOTS)
in_pool = uncensored["in_train_pool"]

train = uncensored[in_train_years & in_pool]           # early years, train-pool loans
test = uncensored[in_test_years & ~in_pool]            # late years, the other loans (~ means "not")
stress = uncensored[in_stress_years & ~in_pool]        # pandemic years, the other loans

# 4. The two split rules.
overlap = set(train["loan_id"]) & set(test["loan_id"])     # loans that are in both train and test
assert len(overlap) == 0, f"{len(overlap)} loans in both train and test"
assert train["snapshot"].max() < test["snapshot"].min()     # every train December is before every test December

print(f"Train rows {len(train):,} (events {train['y'].sum()}) | Test rows {len(test):,} "
      f"(events {test['y'].sum()}) | loan overlap = {len(overlap)}")

**What:** Summarizes each set: the rows before and after dropping the censored rows, the events, the event rate,
and the number of distinct loans.

**Why:** It shows how much data each step removes. It also shows that test and stress have many fewer distinct loans
than rows, because the same loan appears in several Decembers; that matters for the bootstrap in section 9.

**Check:** train 101,843 rows / 462 events (0.454%); test 15,716 / 119 (0.757%); stress 12,445 / 422 (3.391%).

In [ ]:
# For each set: its Decembers, and whether it uses the train-pool loans.
SET_DECEMBERS = {"train": TRAIN_SNAPSHOTS, "test_2021_2024": TEST_SNAPSHOTS, "stress_2019_2020": STRESS_SNAPSHOTS}
SET_USES_POOL_LOANS = {"train": True, "test_2021_2024": False, "stress_2019_2020": False}

summary_rows = []
for set_name in SET_DECEMBERS:
    decembers = SET_DECEMBERS[set_name]
    uses_pool_loans = SET_USES_POOL_LOANS[set_name]

    # The set's rows before dropping the censored rows ...
    in_set_before = snapshots["snapshot"].isin(decembers) & (snapshots["in_train_pool"] == uses_pool_loans)
    rows_before = snapshots[in_set_before]
    # ... and after.
    in_set_after = uncensored["snapshot"].isin(decembers) & (uncensored["in_train_pool"] == uses_pool_loans)
    rows_kept = uncensored[in_set_after]

    summary_rows.append({
        "set": set_name,
        "rows before dropping censored": len(rows_before),
        "censored rows dropped": len(rows_before) - len(rows_kept),
        "rows used": len(rows_kept),
        "events": int(rows_kept["y"].sum()),
        "event rate %": round(100 * rows_kept["y"].mean(), 3),
        "distinct loans": rows_kept["loan_id"].nunique(),
    })

summary = pd.DataFrame(summary_rows)
display(summary.set_index("set"))

## 8. The logistic regression model

For each row $i$ (one loan at one December), the model computes a score $z_i$ and turns it into a probability $p_i$:

$$ z_i = b_0 + \mathbf{x}_i^{\top}\mathbf{b} \qquad\qquad p_i = \sigma(z_i) = \frac{1}{1 + e^{-z_i}} $$

The coefficients are chosen to make the following total as small as possible (the *L2-penalized log-loss*):

$$ \min_{b_0,\ \mathbf{b}} \;\; -\sum_{i=1}^{n} \Big[\, y_i \log p_i + (1 - y_i) \log (1 - p_i) \,\Big]
   \;+\; \frac{1}{2C}\, \lVert \mathbf{b} \rVert_2^2 $$

**The first line, in plain words.**

* $\mathbf{x}_i$ is the row's list of inputs after preprocessing (standardized numbers and 0/1 category columns; see
  below).
* $\mathbf{b}$ holds one coefficient (a weight) per input; $b_0$ is the intercept (a starting value).
* $\mathbf{x}_i^{\top}\mathbf{b}$ means: multiply each input by its weight and add everything up,
  $x_{i1} b_1 + x_{i2} b_2 + \dots$
* $z_i$ is the **log-odds** of an event, $\log\frac{p_i}{1-p_i}$ (the *odds* are $p/(1-p)$; for example $p = 0.2$
  gives odds $0.2/0.8 = 0.25$). $z_i$ can be any number, negative or positive.
* $\sigma(z) = 1/(1+e^{-z})$, the *sigmoid*, squeezes any $z$ into a probability between 0 and 1. $z = 0$ gives
  $p = 0.5$; a very negative $z$ gives a $p$ close to 0. Each $+1$ in $z$ multiplies the odds by $e \approx 2.72$.

**The second line, in plain words.**

* $y_i$ is the observed label: 1 = event, 0 = non-event.
* For an event ($y_i = 1$) the bracket is $\log p_i$; for a non-event ($y_i = 0$) it is $\log(1 - p_i)$. With the
  minus sign in front, a row adds a **small** amount when the model gave the true outcome a high probability, and a
  **large** amount when it did not. Summed over all $n$ rows, this is the **log-loss**.
* $\lVert \mathbf{b} \rVert_2^2 = \sum_j b_j^2$ is the sum of the squared weights. Adding it (the **L2 penalty**)
  discourages large weights. $C$ (`C = 1.0` in SETTINGS) sets the strength: a smaller $C$ makes $\frac{1}{2C}$
  larger, so the weights are pulled more strongly toward 0. The intercept $b_0$ is not penalized.
* "min" means: choose $b_0$ and $\mathbf{b}$ so that the total is as small as possible. scikit-learn finds them step
  by step with the `lbfgs` method.

scikit-learn's `LogisticRegression` with the `lbfgs` solver minimizes exactly this (it multiplies the whole total by
$C$, which does not change the best coefficients). There is **no class weighting**, so $p_i$ stays on the real
event-rate scale (about 0.5%), which the Brier score and the calibration check need. Benchmark pair 2 plants a sign
error in the formula for $p_i$.

The next two small cells put numbers into the two lines.

**What:** Computes $p = 1/(1+e^{-z})$ for a few scores $z$.

**Why:** To see how the sigmoid turns a score into a probability. Our loans have $p$ around 0.5%, so their $z$ is
around $-5$.

**Check:** $z = 0$ gives 0.5000; $z = -5$ gives 0.0067 (0.67%). A larger $z$ always gives a larger $p$.

In [ ]:
for z in [-7, -5, -3, 0, 2]:
    p = 1 / (1 + np.exp(-z))
    print(f"z = {z:3d}   ->   p = {p:.4f}   ({p:.2%})")

**What:** Computes the log-loss of one row, $-[\,y \log p + (1-y)\log(1-p)\,]$, for four cases.

**Why:** To see what fitting pushes the model to do: give events a high $p$ and non-events a low $p$.

**Check:** an event with $p = 0.9$ adds 0.105, but an event with $p = 0.01$ adds 4.605. A non-event with $p = 0.01$
adds only 0.010.

In [ ]:
for y, p in [(1, 0.9), (1, 0.01), (0, 0.01), (0, 0.9)]:
    loss = -(y * np.log(p) + (1 - y) * np.log(1 - p))
    print(f"y = {y}, p = {p:4.2f}   ->   log-loss = {loss:.3f}")

**What:** Lists the 16 numeric and 7 categorical features: the model's inputs.

**Why:** These two lists are the only place where the inputs are chosen. To remove a feature, edit them here (see
the section 13 table).

**Check:** nothing is printed. `current_upb` is not in the lists: it is in the snapshot, but it is not a feature.

In [ ]:
numeric_features = ["fico", "orig_ltv", "orig_dti", "dti_missing", "orig_interest_rate", "log_orig_upb",
                    "term_360", "multi_borrower", "mi_flag", "num_units", "loan_age", "max_dq_12m",
                    "n_dq_months_12m", "ever_90_before_t", "ever_modified", "disaster_flag_t"]

categorical_features = ["status_t", "loan_purpose", "occupancy_status", "first_time_homebuyer",
                        "property_type", "channel", "assist_plan_t"]

**The preprocessing.** Logistic regression needs complete numbers on comparable scales.

* **Numeric features.** First, missing values (for example the 4,020 DTIs that were code 999) are filled with the
  **training median** (the middle value). Then each feature is *standardized*: $(x - \text{mean}) / \text{sd}$, with
  the **training** mean and standard deviation (sd). Example: if the training mean FICO is 750 and its sd is 48, a
  FICO of 702 becomes $(702 - 750) / 48 = -1.0$, "one sd below the mean". (The real training values, 749.8 and
  47.7, are shown at the end of this section.) On standardized features the L2 penalty
  treats every feature equally, and the coefficients can be compared with each other (section 10).
* **Categorical features.** *One-hot encoding* gives one 0/1 column per category (also called a *level*). Example
  for `status_t`: `00` → (1, 0, 0), `01` → (0, 1, 0), `02` → (0, 0, 1), in the columns `status_t_00`, `status_t_01`,
  `status_t_02`. Categories seen fewer than 20 times in training (`min_frequency=20`) share one
  `infrequent_sklearn` column. A category never seen in training gets all zeros (`handle_unknown="ignore"`).

**Why fit the preprocessing on the training rows only?** The medians, means, sds and category lists are part of the
model. If they were computed on test data, information from the test years would leak into the model, and in real
use the future data would not exist yet. Putting everything in one `Pipeline` and calling `fit` only on `train`
guarantees this; on test data, `predict_proba` then *applies* the training values without refitting them.

**What:** Builds the preprocessing and the model, piece by piece, and joins them in one scikit-learn `Pipeline`.
The penalty `C` comes from SETTINGS (the reference uses `C = 1.0`).

**Why:** Keeping preprocessing and model in one object makes it impossible to fit the preprocessing on the wrong
data.

**Check:** nothing is printed.

In [ ]:
# Numeric features: fill missing values with the training median, then standardize.
numeric_steps = Pipeline([
    ("impute", SimpleImputer(strategy="median")),     # missing -> training median
    ("scale", StandardScaler()),                      # (x - training mean) / training sd
])

# Categorical features: one 0/1 column per category; categories seen < 20 times in training share one column.
categorical_step = OneHotEncoder(handle_unknown="ignore", min_frequency=20)

# Apply each part to its own columns. The names "num" and "cat" become the prefixes num__ and cat__ (section 10).
preprocessing = ColumnTransformer([
    ("num", numeric_steps, numeric_features),
    ("cat", categorical_step, categorical_features),
])

# The logistic regression itself.
logistic_regression = LogisticRegression(C=C, max_iter=5000, random_state=SEED)

# Preprocessing + model in one object: model.fit() fits both, on the same (training) rows.
model = Pipeline([
    ("pre", preprocessing),
    ("lr", logistic_regression),
])

**What:** Fits the pipeline on the training rows only.

**Why:** This one `fit` call learns every number of the model: medians, means, sds, categories and coefficients.

**Check:** the solver needs far fewer steps (iterations) than `max_iter = 5000`. It takes a second or two.

In [ ]:
X_train = train[numeric_features + categorical_features]     # the model inputs, one row per loan-December
y_train = train["y"]                                         # the labels: 1 = event, 0 = non-event

model.fit(X_train, y_train)                                  # preprocessing + model, fitted on TRAIN only

n_iterations = model.named_steps["lr"].n_iter_[0]
print("lbfgs iterations:", n_iterations, "(max_iter = 5000)")

**What:** Shows what the preprocessing learned from the training rows: the missing counts, medians, means and sds,
and the categories, including the ones grouped as infrequent.

**Why:** These are the training-only values the text above talks about. They are applied unchanged to the test and
stress data.

**Check:** `orig_dti` has 9,193 missing values in the training rows, filled with the training median 35. Grouped as
infrequent: the `9` of `first_time_homebuyer` (section 3) and the rare assistance plans `R` and `T`.

In [ ]:
# The fitted parts of the pipeline.
fitted_preprocessing = model.named_steps["pre"]
fitted_numeric_steps = fitted_preprocessing.named_transformers_["num"]
fitted_imputer = fitted_numeric_steps.named_steps["impute"]
fitted_scaler = fitted_numeric_steps.named_steps["scale"]
fitted_encoder = fitted_preprocessing.named_transformers_["cat"]

# Numeric features: one row per feature.
learned = pd.DataFrame(index=numeric_features)
learned["missing in train"] = train[numeric_features].isna().sum()
learned["train median (fills missing)"] = fitted_imputer.statistics_
learned["train mean"] = fitted_scaler.mean_
learned["train sd"] = fitted_scaler.scale_
display(learned.round(3))

# Categorical features: the categories found in the training rows, and the ones grouped as infrequent.
for position in range(len(categorical_features)):
    feature = categorical_features[position]
    levels = list(fitted_encoder.categories_[position])
    rare_levels = fitted_encoder.infrequent_categories_[position]
    if rare_levels is None:
        rare_levels = []                                 # nothing was grouped
    else:
        rare_levels = list(rare_levels)
    print(f"{feature:22s} levels: {levels}   grouped as infrequent: {rare_levels}")

## 9. Evaluation

Three metrics, each answering a different question:

* **ROC-AUC (ranking).** Pick one loan that went 90+ and one that did not. The ROC-AUC is the chance that the model
  gave the bad loan the higher risk. 0.5 = a coin flip, 1.0 = a perfect ranking.
* **PR-AUC (average precision: finding the rare bad loans).** If we flag the loans with the highest risk, how many
  of the flagged loans really go 90+ (the *precision*)? PR-AUC averages this over all cut-offs. A model that guesses
  at random scores about the event rate (here under 1%), so read PR-AUC against that base rate, not against 1.
* **Brier score (how good the probabilities are).** The mean squared gap between the predicted probability and what
  happened (0 or 1): $\text{Brier} = \frac{1}{n}\sum_i (p_i - y_i)^2$. Lower is better. Example: a loan with
  $p = 0.02$ and no event adds $(0.02 - 0)^2 = 0.0004$; a loan with $p = 0.02$ and an event adds
  $(0.02 - 1)^2 = 0.96$. It is compared with a "base-rate model" that predicts the set's own event rate for every
  loan. That comparison is generous to the base-rate model, because it uses the true event rate of the very set
  being scored.

**The loan bootstrap (95% confidence intervals).** A *confidence interval* is a range that shows how much a metric
could change by chance. The *bootstrap* estimates it by redrawing the data many times. The test set has 15,716 rows
but only 4,429 distinct loans (section 7), because a loan can appear in up to four test Decembers. Rows of the same
loan are not independent: a weak borrower tends to be weak in every year. So the baseline redraws **loans**, not
rows, each time with all their rows.

A small example with 3 loans A, B, C: one redraw might give A, A, C (A twice, B not at all). Then the metric is
computed on all rows of A (twice) and of C. This is repeated `N_BOOT` = 1000 times. The 95% interval runs from the
2.5th to the 97.5th percentile of the 1000 values (the value with 2.5% of the results below it, and the value with
2.5% above it). A redraw that happens to contain no event at all is skipped, because the metric cannot be computed
on it.

**What:** Defines `cluster_bootstrap_ci()`, the loan bootstrap described above.

**Why:** It gives the uncertainty ranges (95% CIs) that tell us whether a difference between two models is real.

**Check:** nothing is printed. `rng = np.random.default_rng(seed)` makes a new random generator, started from the
seed, at every call. So the intervals are exactly the same in every run.

In [ ]:
def cluster_bootstrap_ci(scored_set, metric, n_resamples, seed):
    """95% confidence interval of a metric, redrawing whole loans (all rows of a loan together)."""
    # A random generator that always starts from the same seed.
    rng = np.random.default_rng(seed)

    # The distinct loans, in the order they first appear.
    loan_ids = scored_set["loan_id"].unique()

    # For each loan, the positions of its rows. (The set's rows are numbered 0 .. n-1; see add_predictions().)
    rows_of_loan = {}
    for loan_id, loan_rows in scored_set.groupby("loan_id"):
        rows_of_loan[loan_id] = loan_rows.index.values

    all_y = scored_set["y"].values          # the labels, as a plain array
    all_p = scored_set["p"].values          # the predicted probabilities

    metric_values = []
    for resample_number in range(n_resamples):
        # 1. Draw as many loans as there are, with replacement (a loan can be drawn twice, or not at all).
        drawn_loans = rng.choice(loan_ids, size=len(loan_ids), replace=True)

        # 2. Collect the row positions of every drawn loan into one array.
        drawn_row_lists = []
        for loan_id in drawn_loans:
            drawn_row_lists.append(rows_of_loan[loan_id])
        drawn_rows = np.concatenate(drawn_row_lists)

        # 3. The labels and predictions of those rows.
        y = all_y[drawn_rows]
        p = all_p[drawn_rows]

        # 4. Skip a redraw with only one kind of label (no events, or no non-events): the metric is undefined.
        if y.min() == y.max():
            continue
        metric_values.append(metric(y, p))

    # The 2.5th and 97.5th percentiles of the metric values.
    lower = float(np.percentile(metric_values, 2.5))
    upper = float(np.percentile(metric_values, 97.5))
    return [lower, upper]

**What:** Defines `format_with_ci()`, which writes a value and its interval as one text, and `evaluate()`, which
computes every number stored in `reference/metrics.json` for one set and prints a two-line summary. The bootstrap
uses `N_BOOT` and `SEED` from SETTINGS (the reference used 1000 resamples).

**Why:** `evaluate()` runs three times (train, test, stress), so it is a function. The order of the items in the
result is the order of `metrics.json`.

**Check:** nothing is printed until the evaluation cell below runs it.

In [ ]:
def format_with_ci(value, ci):
    """Write a value and its 95% interval as text, for example '0.894 [0.849, 0.930]'."""
    return f"{value:.3f} [{ci[0]:.3f}, {ci[1]:.3f}]"


def evaluate(scored_set, name):
    """Compute the metrics of one scored set (the items of reference/metrics.json) and print a summary."""
    y = scored_set["y"].values
    p = scored_set["p"].values

    # The "base-rate model": the same prediction, the set's event rate, for every row.
    base_rate_prediction = np.full_like(p, y.mean())

    result = {}
    result["n"] = int(len(y))
    result["events"] = int(y.sum())
    result["event_rate"] = float(y.mean())
    result["mean_predicted"] = float(p.mean())
    result["roc_auc"] = float(roc_auc_score(y, p))
    result["pr_auc"] = float(average_precision_score(y, p))
    result["brier"] = float(brier_score_loss(y, p))
    result["brier_reference_base_rate"] = float(brier_score_loss(y, base_rate_prediction))
    result["roc_auc_95ci"] = cluster_bootstrap_ci(scored_set, roc_auc_score, N_BOOT, SEED)
    result["pr_auc_95ci"] = cluster_bootstrap_ci(scored_set, average_precision_score, N_BOOT, SEED)

    print(f"\n[{name}] n={result['n']:,} events={result['events']} rate={result['event_rate']:.4f} "
          f"mean_p={result['mean_predicted']:.4f}")
    print(f"  ROC-AUC {format_with_ci(result['roc_auc'], result['roc_auc_95ci'])} | "
          f"PR-AUC {format_with_ci(result['pr_auc'], result['pr_auc_95ci'])} | "
          f"Brier {result['brier']:.5f} (base-rate model {result['brier_reference_base_rate']:.5f})")
    return result

**What:** Defines `add_predictions()` and uses it to score the train, test and stress sets: each row gets `p`, the
predicted probability of an event.

**Why:** The metrics compare `p` with `y`. The rows are renumbered 0 … n−1, because the bootstrap finds the rows of
each loan by their number.

**Check:** the mean predicted probability: about 0.46% (train), 0.43% (test), 0.61% (stress).

In [ ]:
def add_predictions(data_set):
    """Return a copy of a set with the predicted probability p added, and its rows numbered 0 .. n-1."""
    scored_set = data_set.copy()
    model_inputs = scored_set[numeric_features + categorical_features]
    probabilities = model.predict_proba(model_inputs)     # two columns: P(y = 0) and P(y = 1)
    scored_set["p"] = probabilities[:, 1]                  # keep P(y = 1), the chance of an event
    scored_set = scored_set.reset_index(drop=True)         # number the rows 0 .. n-1
    return scored_set


scored_train = add_predictions(train)
scored_test = add_predictions(test)
scored_stress = add_predictions(stress)

print(f"Mean predicted probability: train {scored_train['p'].mean():.2%}, "
      f"test {scored_test['p'].mean():.2%}, stress {scored_stress['p'].mean():.2%}")

**What:** Evaluates each set (metrics + bootstrap intervals). Nothing is written to disk here; with
`SAVE_OUTPUTS = True`, the last cell of section 12 saves the files.

**Why:** These are the headline results of the baseline.

**Check:** test: ROC-AUC 0.894 [0.849, 0.930], PR-AUC 0.307, Brier 0.00620. Train ROC-AUC 0.890; stress 0.732. This is
the slow cell; the time is printed at the end.

In [ ]:
start_time = time.perf_counter()

results = {}
results["train"] = evaluate(scored_train, "train")
results["test_2021_2024"] = evaluate(scored_test, "test_2021_2024")
results["stress_2019_2020"] = evaluate(scored_stress, "stress_2019_2020")

seconds = time.perf_counter() - start_time
print(f"\n(evaluation took {seconds:.0f} s with N_BOOT = {N_BOOT})")

**What:** Puts the results of the three sets side by side in one table.

**Why:** It is easier to compare train, test and stress side by side than in the printed lines.

**Check:** test PR-AUC 0.307 is about 40 times the test event rate (0.76%). The Brier score is below the base-rate
model in all three sets. Mean predicted vs event rate: almost equal in train; lower than observed in test and
stress.

In [ ]:
SET_NAMES = ["train", "test_2021_2024", "stress_2019_2020"]

summary_table = pd.DataFrame()
for set_name in SET_NAMES:
    set_results = results[set_name]
    column = {
        "rows": f"{set_results['n']:,}",
        "events": set_results["events"],
        "event rate": f"{set_results['event_rate']:.2%}",
        "mean predicted": f"{set_results['mean_predicted']:.2%}",
        "ROC-AUC [95% CI]": format_with_ci(set_results["roc_auc"], set_results["roc_auc_95ci"]),
        "PR-AUC [95% CI]": format_with_ci(set_results["pr_auc"], set_results["pr_auc_95ci"]),
        "Brier (base-rate model)": f"{set_results['brier']:.5f} ({set_results['brier_reference_base_rate']:.5f})",
    }
    summary_table[set_name] = pd.Series(column)
display(summary_table)

**What:** Builds the test calibration table: it sorts the test rows into 10 equal-sized groups (*deciles*) by
predicted risk, and compares the mean prediction with the observed event rate in each group. The reference copy is
`reference/calibration.csv`.

**Why:** A model can rank well (high AUC) and still predict the wrong *level* of risk. Calibration checks the level.

**Check:** 10 rows (bin 0 = lowest risk … bin 9 = highest), about 1,572 rows each. The top decile: predicted about
3.6%, observed about 5.8%.

In [ ]:
calibration_data = scored_test.copy()

# Rank the rows by p (1 = lowest). method="first": equal values get different ranks, in row order.
risk_rank = calibration_data["p"].rank(method="first")

# Cut the ranks into 10 equal-sized groups: bin 0 = lowest risk, bin 9 = highest risk.
calibration_data["bin"] = pd.qcut(risk_rank, 10, labels=False)

# One row per bin: rows, mean prediction, observed event rate, events.
groups_by_bin = calibration_data.groupby("bin")
calibration = pd.DataFrame({
    "n": groups_by_bin["y"].size(),
    "mean_predicted": groups_by_bin["p"].mean(),
    "observed_rate": groups_by_bin["y"].mean(),
    "events": groups_by_bin["y"].sum(),
})
calibration = calibration.reset_index()      # make "bin" a normal column
display(calibration.round(5))

**What:** Defines `plot_calibration()` and draws the calibration table twice: on the left all 10 deciles (the
original script's plot, saved as `outputs/calibration.png` when `SAVE_OUTPUTS = True`); on the right a zoom on the 9
lowest deciles, which are squeezed into the corner of the left plot.

**Why:** Points on the dashed diagonal mean that the predicted risk equals the observed rate. Points above it mean
that the model predicts too little risk.

**Check:** 8 of the 10 points lie above the diagonal. In the zoom the points jump up and down, because each low
decile holds only 0–10 events: that is noise, not a pattern.

In [ ]:
def plot_calibration(ax, table, title):
    """Draw one point per decile (mean predicted vs observed rate) and the diagonal line of perfect calibration."""
    ax.plot(table["mean_predicted"], table["observed_rate"], "o-", color="black", label="model")
    largest_value = max(table["mean_predicted"].max(), table["observed_rate"].max())
    line_end = largest_value * 1.05
    ax.plot([0, line_end], [0, line_end], "--", color="grey", label="perfect")
    ax.set_xlabel("Mean predicted probability")
    ax.set_ylabel("Observed 90+ rate")
    ax.set_title(title)
    ax.legend(frameon=False)


figure, axes = plt.subplots(1, 2, figsize=(9.5, 4.5))
left_panel = axes[0]
right_panel = axes[1]

lowest_9_deciles = calibration[calibration["bin"] < 9]
plot_calibration(left_panel, calibration, "Calibration, test 2021-2024")
plot_calibration(right_panel, lowest_9_deciles, "Zoom: the 9 lowest deciles")

# A light grid, behind the points.
for panel in axes:
    panel.grid(color="0.92")
    panel.set_axisbelow(True)

figure.tight_layout()
plt.show()                 # shown only; section 12 saves the left plot if SAVE_OUTPUTS = True

**Why the model under-predicts on test (mean predicted 0.43% vs observed 0.76%), and why that is not a code
defect.** On the training rows, the mean prediction (0.456%) almost equals the observed rate (0.454%). That is a
property of logistic regression with an unpenalized intercept: at the best coefficients, the predictions add up to
the number of events in the training data. So the model's overall *level* is tied to the 2015–2018 event rate.

The test years 2021–2024 simply have a higher 90+ rate (section 6: about 0.67–0.90% per year, against 0.24–0.65% in
the training years). The population and the economy changed between the two periods: a **distribution shift**. The
ranking still holds up well (test ROC-AUC 0.894, about the same as on train), so the model still separates riskier
from safer loans; only its level is off.

The code does exactly what the specification says. As `CLAUDE.md` states, distribution shift and imperfect accuracy
are not implementation defects. Fixing the level (recalibrating on recent data) would be a modelling change, not a
bug fix, and it is not part of the baseline.

## 10. Coefficients

**What:** Builds the coefficient table, `coefficients` (one row per input column, plus the intercept; the reference
copy is `reference/coefficients.csv`). Then it shows a copy sorted by size, with the odds ratio $e^{b}$ of each
numeric feature.

**Why:** The coefficients show what the model has learned, and section 12 compares them with the reference.

**Check:** 40 rows (39 input columns + the intercept). The largest are the `status_t` levels; among the numeric
features: `fico` −0.375, `max_dq_12m` +0.332, `multi_borrower` −0.289, `orig_ltv` +0.256.

In [ ]:
fitted_model = model.named_steps["lr"]

# The names of the 39 input columns after preprocessing (num__... and cat__...), and their coefficients.
feature_names = list(model.named_steps["pre"].get_feature_names_out())
coefficient_values = list(fitted_model.coef_[0])

# Add the intercept as the last row.
feature_names.append("intercept")
coefficient_values.append(fitted_model.intercept_[0])

coefficients = pd.DataFrame({"feature": feature_names, "coef_standardized": coefficient_values})

# A copy for reading: sorted by size (largest first, ignoring the sign), rounded to 3 decimals.
coef_view = coefficients.copy()
coef_view["size"] = coef_view["coef_standardized"].abs()
coef_view = coef_view.sort_values("size", ascending=False)
coef_view = coef_view.drop(columns="size")
coef_view = coef_view.set_index("feature")
coef_view = coef_view.round(3)

# The odds ratio e^b, only for the numeric features (their names start with "num__"); empty (NaN) for the others.
is_numeric_feature = coef_view.index.str.startswith("num__")
odds_ratios = np.exp(coef_view["coef_standardized"]).round(3)
coef_view["odds ratio per +1 sd"] = np.nan
coef_view.loc[is_numeric_feature, "odds ratio per +1 sd"] = odds_ratios[is_numeric_feature]

display(coef_view)

**How to read standardized coefficients.** Every numeric feature was standardized with the training mean and sd.
So a coefficient is the change in log-odds for a **one-sd increase** in that feature, with all other features held
fixed.

Example: `fico` = −0.375 means that one sd more FICO multiplies the odds of going 90+ by $e^{-0.375} \approx 0.69$,
a 31% reduction. Because all numeric features share this scale, their sizes can be compared with each other.

For a 0/1 flag, "one sd" is only a part of the step from 0 to 1, so the full 0 → 1 effect is $b / \text{sd}$, with
the sd from section 8. Coefficients describe associations, given the other features, not causes. Related features
share their effect: `max_dq_12m`, `n_dq_months_12m` and `status_t` all describe recent delinquency, so the small
negative values of `n_dq_months_12m` and `ever_90_before_t` should not be read on their own. `CLAUDE.md` is explicit
that odd-looking coefficients are not implementation defects.

**The one-hot warning: compare levels with each other, not with zero.** The encoder keeps **all** levels of each
categorical feature (no reference level is dropped). Each row has exactly one level set to 1, so the levels of one
feature always add up to a column of 1s, just like the intercept. The model could move any amount between the
intercept and the levels without changing a single prediction, and the L2 penalty decides how it is split. So one
level's coefficient compared with 0 means nothing (for example, all `assist_plan_t` levels are negative). Only
**differences between levels of the same feature** mean something. The next cell computes them for `status_t`.
Exercise 3 drops a reference level instead.

**What:** For each `status_t` level, computes its difference from the `00` (current) level, and the matching odds
ratio.

**Why:** This is the right way to read one-hot coefficients, and the status at $t$ is the strongest predictor in the
model.

**Check:** `01` vs `00`: about +1.85 (odds about 6× higher); `02` vs `00`: about +3.04 (odds about 21× higher), with
the other features held fixed.

In [ ]:
coefficient_of = coefficients.set_index("feature")["coef_standardized"]     # look up a coefficient by name

is_status_level = coefficient_of.index.str.startswith("cat__status_t_")
status_coefficients = coefficient_of[is_status_level]
current_level = coefficient_of["cat__status_t_00"]           # the current (00) level is the comparison

difference = status_coefficients - current_level
status_table = pd.DataFrame({
    "coefficient": status_coefficients.round(3),
    "difference vs status 00": difference.round(3),
    "odds ratio vs status 00": np.exp(difference).round(2),
})
display(status_table)

## 11. Leakage check

**Future invariance, in plain words.** A feature at prediction month $t$ must be something you could really have
known on 31 December of that year. If that is true, then **deleting or changing anything after $t$ cannot change any
feature**. If some feature does change, it has looked into the future: that is **leakage**. Leakage makes a model
look much better in a backtest than it can ever be in real use. The **label**, on the other hand, *must* change,
because it is defined on the months $t+1 \dots t+12$.

The checks below run on the toy table from section 5, in two ways: **truncation** (delete every month after $t$) and
**perturbation** (keep the months after $t$, but change their values). Each time we rerun `build_snapshot()` and
compare the features with the original ones.

**What:** Defines `feature_table()`, then builds the toy snapshot twice: once from the full toy table, once from the
toy table with every month after $t$ deleted. It compares all feature columns.

**Why:** This is the general leakage test. It does not need to know *how* a leak might happen: any feature that reads
the future fails it.

**Check:** `Same loans: True` and `Feature columns identical ...: True`. The labels do change: with no future, every
loan becomes `censored_end_of_data`.

In [ ]:
def feature_table(snapshot_table):
    """The snapshot with one row per loan (sorted by loan ID) and only the feature columns (no label, no month)."""
    table = snapshot_table.set_index("loan_id")
    table = table.sort_index()
    table = table.drop(columns=["label", "snapshot"])
    return table


t_toy = int(ym_index([T_TOY])[0])

# The toy table without the months after t.
toy_until_t = toy[toy["t"] <= t_toy]

snapshot_full = build_snapshot(toy, T_TOY)               # all 30 months
snapshot_cut = build_snapshot(toy_until_t, T_TOY)        # months after t deleted

features_full = feature_table(snapshot_full)
features_cut = feature_table(snapshot_cut)

# .equals() is True only if both tables have the same rows, columns, values and types.
print("Same loans:", list(features_full.index) == list(features_cut.index))
print("Feature columns identical after deleting every month after t:", features_full.equals(features_cut))

labels = pd.DataFrame({
    "label (full table)": snapshot_full.set_index("loan_id")["label"],
    "label (months after t deleted)": snapshot_cut.set_index("loan_id")["label"],
})
display(labels)

**What:** Builds the toy snapshot from a *perturbed* (changed) toy table. Every month after $t$ is kept, but its
status becomes `05` (150–179 days late), and its modification, disaster and assistance fields become `Y`, `Y` and
`F`. Then `add_derived_columns()` recomputes `dq` and `sev`. All feature columns are compared with the original
ones.

**Why:** Truncation shows that *deleting* the future changes no feature. Perturbation shows that *changing* it changes
none either: a feature that read the future would pick up the new values. This is test T2 of
`tests/test_baseline_no_leakage.py`, run on the toy table.

**Check:** `Same loans: True` and `Feature columns identical ...: True`. The labels do change: every at-risk loan now
has a 90+ month in its window, so all four become `event` (`X03` too: its records 201801–201804 lie after $t$).

In [ ]:
is_after_t = toy["t"] > t_toy                                 # every record after the prediction month

perturbed = toy.copy()
perturbed.loc[is_after_t, "delinq_status"] = "05"             # 150-179 days late in every month after t ...
perturbed.loc[is_after_t, "modification_flag"] = "Y"          # ... and modified,
perturbed.loc[is_after_t, "delinq_due_to_disaster"] = "Y"     # ... with a disaster flag,
perturbed.loc[is_after_t, "borrower_assistance_plan"] = "F"   # ... and in forbearance
add_derived_columns(perturbed)                                # recompute t, dq and sev from the new status

snapshot_perturbed = build_snapshot(perturbed, T_TOY)
features_perturbed = feature_table(snapshot_perturbed)

print("Same loans:", list(features_full.index) == list(features_perturbed.index))
print("Feature columns identical after changing every month after t:", features_full.equals(features_perturbed))

labels = pd.DataFrame({
    "label (original table)": snapshot_full.set_index("loan_id")["label"],
    "label (months after t changed)": snapshot_perturbed.set_index("loan_id")["label"],
})
display(labels)

**What:** A demonstration of the *power check*. It makes a deliberately leaky copy of `build_snapshot()`, in which the
12-month lookback is taken from **all** records instead of only those up to $t$, and runs the truncation comparison
on it.

How the copy is made: `inspect.getsource()` reads the text of `build_snapshot()`; `.replace()` swaps the `h12` line
for a leaky one; `exec()` runs the changed text as Python code, which defines the leaky function. It runs in a copy
of the notebook's variables (`dict(globals())`), so the real `build_snapshot()` is not overwritten.

**Why:** A test that can never fail proves nothing. An earlier version of the baseline had exactly such a check (an
assertion that could never fail). Showing that the comparison catches a real leak is what makes the "identical"
results above meaningful.

**Check:** `Leaky copy, features identical after truncation: False`. `X02` and `X05` get `max_dq_12m` = 3 from the
full table, because the leaky lookback sees their 90+ months after $t$.

In [ ]:
GOOD_LINE = "h12 = hist[(hist.t > t - 12) & hist.loan_id.isin(ids)]"     # lookback from hist (months <= t)
LEAKY_LINE = "h12 = perf[(perf.t > t - 12) & perf.loan_id.isin(ids)]"    # the same window from perf: a leak

original_source = inspect.getsource(build_snapshot)

if original_source.count(GOOD_LINE) != 1:
    print("build_snapshot() was edited, so this demonstration's text change no longer applies; skipped.")
else:
    leaky_source = original_source.replace(GOOD_LINE, LEAKY_LINE)
    leaky_namespace = dict(globals())             # a copy of the notebook's variables
    exec(leaky_source, leaky_namespace)           # defines the leaky build_snapshot inside that copy
    leaky_build_snapshot = leaky_namespace["build_snapshot"]

    leaky_full = feature_table(leaky_build_snapshot(toy, T_TOY))
    leaky_cut = feature_table(leaky_build_snapshot(toy_until_t, T_TOY))
    print("Leaky copy, features identical after truncation:", leaky_full.equals(leaky_cut))

    leaky_comparison = pd.DataFrame({
        "max_dq_12m (full table)": leaky_full["max_dq_12m"],
        "max_dq_12m (after t deleted)": leaky_cut["max_dq_12m"],
        "n_dq_months_12m (full table)": leaky_full["n_dq_months_12m"],
        "n_dq_months_12m (after t deleted)": leaky_cut["n_dq_months_12m"],
    })
    display(leaky_comparison)

**The formal tests.** `tests/test_baseline_no_leakage.py` runs these checks on this notebook's own
`build_snapshot()`. It reads the two code cells tagged `leakage-test-source` (the cells that define `ym_index()` in
section 4 and `build_snapshot()` in section 5) and runs them in a fresh Python namespace that holds only `np`, `pd`
and `HORIZON = 12` (the outcome window of the specification; in this notebook `HORIZON` comes from the SETTINGS
cell). It then tests the result on eight made-up loans, `T01`–`T08`: T1 truncation, T2 perturbation (changing the
status, modification, disaster and assistance fields after $t$), T3 the power check shown above (applied to the text
of the tagged cell), and T4 hand-computed feature values. Run them from the repository root:

```
uv run pytest tests -v
```

All 4 tests should pass.

**If you edit the two tagged cells**, keep their tag (it is stored in the cell's metadata) and keep them
self-contained: they may use only `np`, `pd`, `HORIZON` and each other. If no cell carries the tag, or if the line
`h12 = hist[(hist.t > t - 12) & hist.loan_id.isin(ids)]` that T3 changes is changed, the tests fail with a message
that says so.

## 12. Does the notebook match the reference?

The official reference results are in `reference/`. They were made by the original script (`src/baseline_logreg.py`
at commit `9253e97`; see `reference/README.md`). This section compares the notebook's results with them, item by
item:

* **label counts** (`label_counts.csv`) and **set sizes** (rows and events per set): must be exactly equal;
* **metrics** (`metrics.json`: event rates, mean predictions, ROC-AUC, PR-AUC, Brier, and the interval ends) and the
  run **configuration**: numbers must agree to within $10^{-9}$ (0.000000001);
* **coefficients** (`coefficients.csv`) and the **calibration table** (`calibration.csv`): within $10^{-9}$.

It never stops with an error. If something differs, it prints `DIFFERS FROM THE REFERENCE` and a table of every
difference (reference = old, notebook = new). That is the expected result after you have changed something on
purpose. With `N_BOOT` other than 1000, only the confidence-interval items (`*_95ci`) should differ.

The last cell writes the result files to `outputs/` when `SAVE_OUTPUTS = True`.

**What:** Adds the run configuration and the library versions to `results`, in the same form as
`reference/metrics.json`.

**Why:** The configuration is part of the reference, so a changed setting appears in the difference table by name
(for example `config.horizon_months: 12 → 6`).

**Check:** the printed configuration matches the SETTINGS cell.

In [ ]:
results["config"] = {
    "seed": SEED,
    "train_snapshots": TRAIN_SNAPSHOTS,
    "test_snapshots": TEST_SNAPSHOTS,
    "horizon_months": HORIZON,
    "train_loan_share_pct": TRAIN_LOAN_SHARE,
    "C": C,
    "features_numeric": numeric_features,
    "features_categorical": categorical_features,
}
results["versions"] = {
    "python": platform.python_version(),
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "scikit-learn": sklearn.__version__,
}

# Print the configuration, without the two long feature lists.
for key in ["seed", "train_snapshots", "test_snapshots", "horizon_months", "train_loan_share_pct", "C"]:
    print(f"{key:22s}", results["config"][key])

**What:** Defines small helpers for the comparison:

* `flatten_results()` turns the nested results into named items, for example
  `{"test_2021_2024": {"roc_auc": 0.894}}` → `{"test_2021_2024.roc_auc": 0.894}`;
* `items_of_part()` picks the items of one part (set sizes, metrics or configuration);
* `label_count_items()`, `coefficient_items()` and `calibration_items()` turn the three tables into named items;
* `compare_items()` lists every item that differs by more than the tolerance, or exists on one side only.

**Why:** Comparing item by item, instead of whole files, tells you exactly *what* changed. That is the point of this
section.

**Check:** nothing is printed.

In [ ]:
TOL = 1e-9                     # numbers closer than this count as equal
REF_DIR = Path("reference")    # the official reference results (read only; see reference/README.md)


def flatten_results(results_dict):
    """{'test': {'roc_auc': 0.89, 'roc_auc_95ci': [a, b]}} -> {'test.roc_auc': 0.89, 'test.roc_auc_95ci[0]': a, ...}"""
    flat = {}
    for group_name, group in results_dict.items():
        if group_name == "versions":
            continue                                  # the library versions are reported separately
        for item_name, value in group.items():
            key = group_name + "." + item_name
            if item_name.endswith("_95ci"):
                flat[key + "[0]"] = value[0]          # a confidence interval -> its two ends
                flat[key + "[1]"] = value[1]
            else:
                flat[key] = value                     # a number, a text, or a list of names or months
    return flat


def part_of(key):
    """The part an item of metrics.json belongs to: 'set sizes' (n, events), 'config', or 'metrics'."""
    last_name = key.split(".")[-1]
    if last_name in ("n", "events"):
        return "set sizes"
    if key.startswith("config."):
        return "config"
    return "metrics"


def items_of_part(flat_results, part):
    """Only the items of one part."""
    chosen = {}
    for key, value in flat_results.items():
        if part_of(key) == part:
            chosen[key] = value
    return chosen


def label_count_items(count_table):
    """{'201512 / event': 95, ...}: one item per cell of the label-count table that is not 0."""
    items = {}
    for snapshot in count_table.index:
        for label in count_table.columns:
            value = int(count_table.loc[snapshot, label])
            if value != 0:                            # a missing cell means 0 loans, so zeros are left out
                items[f"{snapshot} / {label}"] = value
    return items


def coefficient_items(coefficient_table):
    """{'num__fico': -0.3746..., ...}: one item per coefficient."""
    items = {}
    for row in coefficient_table.index:
        feature = coefficient_table.loc[row, "feature"]
        items[feature] = coefficient_table.loc[row, "coef_standardized"]
    return items


def calibration_items(calibration_table):
    """{'bin 0 / n': 1572, 'bin 0 / mean_predicted': 0.00018..., ...}: one item per cell of the calibration table."""
    items = {}
    for row in calibration_table.index:
        bin_number = int(calibration_table.loc[row, "bin"])
        for column in calibration_table.columns:
            if column != "bin":
                items[f"bin {bin_number} / {column}"] = calibration_table.loc[row, column]
    return items


def is_number(x):
    """True for an int or a float (also the numpy kinds), but not for True/False."""
    return isinstance(x, (int, float, np.integer, np.floating)) and not isinstance(x, bool)


def number_of_items(reference_items, notebook_items):
    """How many different item names the two sides have together."""
    all_names = set(reference_items) | set(notebook_items)      # | joins the two sets of names
    return len(all_names)


def compare_items(part, reference_items, notebook_items, missing_value="(not present)"):
    """One row for every item that differs between the reference and the notebook, or exists on one side only."""
    # All item names: first the reference's, then the ones that only the notebook has.
    all_keys = list(reference_items)
    for key in notebook_items:
        if key not in reference_items:
            all_keys.append(key)

    rows = []
    for key in all_keys:
        old = reference_items.get(key, missing_value)     # .get(): the value, or missing_value if there is none
        new = notebook_items.get(key, missing_value)
        both_numbers = is_number(old) and is_number(new)
        if both_numbers:
            same = abs(old - new) <= TOL                  # numbers: equal within the tolerance
        else:
            same = old == new                             # texts and lists: exactly equal
        if not same:
            if both_numbers:
                abs_difference = abs(old - new)
            else:
                abs_difference = None
            rows.append({"part": part, "item": key, "reference (old)": old, "notebook (new)": new,
                         "abs. difference": abs_difference})
    return rows

**What:** Reads the four reference files and compares them, part by part, with the notebook's results. Every
difference is collected in the table `diff`.

**Why:** Each part (set sizes, metrics, configuration, label counts, coefficients, calibration) is compared
separately, so a difference can be traced to one step of the pipeline.

**Check:** nothing is printed; the verdict follows in the next cell.

In [ ]:
# The four reference files.
ref_metrics = json.loads((REF_DIR / "metrics.json").read_text())
ref_label_counts = pd.read_csv(REF_DIR / "label_counts.csv", index_col="snapshot")
ref_coefficients = pd.read_csv(REF_DIR / "coefficients.csv")
ref_calibration = pd.read_csv(REF_DIR / "calibration.csv")

# metrics.json and the notebook's results, as flat named items.
ref_flat = flatten_results(ref_metrics)
new_flat = flatten_results(results)

difference_rows = []     # every difference found
values_compared = {}     # how many items each part compared

# Parts 1-3: set sizes, metrics and configuration (all from metrics.json).
for part in ["set sizes", "metrics", "config"]:
    ref_items = items_of_part(ref_flat, part)
    new_items = items_of_part(new_flat, part)
    difference_rows += compare_items(part, ref_items, new_items)
    values_compared[part] = number_of_items(ref_items, new_items)

# Part 4: label counts. A count that is missing on one side means 0 loans.
ref_items = label_count_items(ref_label_counts)
new_items = label_count_items(label_counts)
difference_rows += compare_items("label counts", ref_items, new_items, missing_value=0)
values_compared["label counts"] = number_of_items(ref_items, new_items)

# Part 5: coefficients.
ref_items = coefficient_items(ref_coefficients)
new_items = coefficient_items(coefficients)
difference_rows += compare_items("coefficients", ref_items, new_items)
values_compared["coefficients"] = number_of_items(ref_items, new_items)

# Part 6: calibration table.
ref_items = calibration_items(ref_calibration)
new_items = calibration_items(calibration)
difference_rows += compare_items("calibration", ref_items, new_items)
values_compared["calibration"] = number_of_items(ref_items, new_items)

diff = pd.DataFrame(difference_rows,
                    columns=["part", "item", "reference (old)", "notebook (new)", "abs. difference"])

**What:** Prints the verdict, the number of values compared and the number that differ in each part, and, if
anything differs, the full table of differences (reference = old, notebook = new).

**Why:** This is the proof that the notebook reproduces the reference exactly, and the place that tells you what your
own edits changed.

**Check:** `MATCHES THE REFERENCE MODEL`, with 0 differences in every part (6 set sizes, 30 metrics, 8 configuration
items, 40 label counts, 40 coefficients, 40 calibration values).

In [ ]:
print("=" * 60)
if diff.empty:
    print("MATCHES THE REFERENCE MODEL")
else:
    print("DIFFERS FROM THE REFERENCE")
print("=" * 60)

# For each part: how many values were compared, and how many of them differ.
differences_per_part = {}
for part in values_compared:
    differences_per_part[part] = int((diff["part"] == part).sum())

overview = pd.DataFrame(index=list(values_compared))
overview["values compared"] = pd.Series(values_compared)
overview["differences"] = pd.Series(differences_per_part)
display(overview)

# The full list of differences (all rows, long texts in full).
if not diff.empty:
    with pd.option_context("display.max_rows", None, "display.max_colwidth", 80):
        display(diff)

if N_BOOT != 1000:
    print(f"Note: N_BOOT = {N_BOOT}, but the reference used 1000 resamples, so the *_95ci items are expected to differ.")
if results["versions"] != ref_metrics.get("versions"):
    print("Note: library versions differ from the reference run:", ref_metrics.get("versions"), "->", results["versions"])

**What:** Only if `SAVE_OUTPUTS = True` (SETTINGS cell): writes the five result files to `outputs/` (gitignored) in the
original script's format: `metrics.json`, `label_counts.csv`, `coefficients.csv`, `calibration.csv` and
`calibration.png` (the one-panel calibration plot). With the default `False` it writes nothing.

**Why:** The first four files have exactly the format of the files in `reference/`; with the default settings they
are byte-identical to them. They are the files to review, and to copy into `reference/` if a change is ever made
official (section 13). The notebook itself never writes to `reference/`.

**Check:** `SAVE_OUTPUTS = False: nothing written.` With `True`: the folder and the five file names.

In [ ]:
if SAVE_OUTPUTS:
    output_folder = Path("outputs")                       # gitignored; reference/ is never written
    output_folder.mkdir(exist_ok=True)

    label_counts.to_csv(output_folder / "label_counts.csv")
    calibration.to_csv(output_folder / "calibration.csv", index=False)
    coefficients.to_csv(output_folder / "coefficients.csv", index=False)
    metrics_text = json.dumps(results, indent=2)          # metrics, configuration and versions
    (output_folder / "metrics.json").write_text(metrics_text)

    # The original script's one-panel calibration plot: saved, not shown.
    figure, ax = plt.subplots(figsize=(4.5, 4.5))
    plot_calibration(ax, calibration, "Calibration, test 2021-2024")
    figure.tight_layout()
    figure.savefig(output_folder / "calibration.png", dpi=150)
    plt.close(figure)

    print(f"Saved to {output_folder.resolve()}: metrics.json, label_counts.csv, coefficients.csv, "
          f"calibration.csv, calibration.png")
else:
    print("SAVE_OUTPUTS = False: nothing written.")

## 13. Where to make changes, and optional exercises

### Safe edit points

| What to change | Where | Rerun sections | Reference numbers that will change (section 12) |
|---|---|---|---|
| Snapshot dates (`TRAIN_SNAPSHOTS`, `TEST_SNAPSHOTS`, `STRESS_SNAPSHOTS`) | SETTINGS cell (§1) | 6 → 12 | label counts (rows added or removed), set sizes, every metric and CI, coefficients, calibration, `config.*_snapshots`. Keep every window inside the data: December + `HORIZON` months ≤ 202603. |
| Horizon (`HORIZON`) | SETTINGS cell (§1) | 5 → 12 (the toy uses it too) | every label, so label counts, set sizes, metrics, coefficients, calibration, `config.horizon_months`. |
| Penalty (`C`) | SETTINGS cell (§1) | 8 → 12 | coefficients, every metric and CI, calibration, `config.C`. Label counts and set sizes stay the same. |
| Bootstrap size (`N_BOOT`) or `SEED` | SETTINGS cell (§1) | 9 → 12 | only the `*_95ci` items (and `config.seed` for `SEED`). |
| Save the result files (`SAVE_OUTPUTS`) | SETTINGS cell (§1) | the last cell of 12 | none: it only writes `outputs/`. |
| Remove a feature | the `numeric_features` / `categorical_features` lists (§8) | 8 → 12 | coefficients (its row disappears), metrics, calibration, `config.features_*`. |
| Add a feature | compute it in §3 (next to the five derived features, and add its name to `LOAN_FEATURE_COLUMNS`) or in `build_snapshot()` (§5, from `hist`/`now` only, i.e. months ≤ $t$); then add its name to `numeric_features` or `categorical_features` (§8) | from the section you edited → 12; then rerun §11 and `uv run pytest tests -v` to confirm there is no leakage | coefficients (a new row), metrics, calibration, `config.features_*`. |
| Censoring policy: prepayment as y = 0 (sensitivity run) | the `is_uncensored = ...` line (§7, the split cell) | 7 → 12 | set sizes (more rows, same events), event rates, metrics, coefficients, calibration. Label counts stay the same (they are counted before this step). |

When in doubt, **Restart** the kernel and **Run All**. Each cell uses the variables made by the cells above it, so
rerunning only a part of the notebook can mix old and new values.

### Optional exercises (not applied; each one is a change you can try)

1. **Map the categorical "9"/"99" codes to an explicit "unknown" category.**
   *Hint:* in section 3, after the cell that replaces the codes with NaN, add a cell:

   ```python
   for field in ["loan_purpose", "occupancy_status", "first_time_homebuyer", "property_type", "channel"]:
       is_unknown = orig_clean[field].isin(["9", "99"])
       orig_clean.loc[is_unknown, field] = "unknown"
   ```

   Check with `value_counts()` that only the 8 `first_time_homebuyer` loans change. Then ask yourself: with
   `min_frequency=20`, does the encoder treat "unknown" any differently from before?
2. **Turn the current balance into a feature: current UPB ÷ original UPB, using only month-$t$ data.**
   *Hint:* `build_snapshot()` already copies `current_upb` from the month-$t$ record (`now`), so it uses no future
   data. Add `"orig_upb"` to `LOAN_FEATURE_COLUMNS` (§3). After the merge in §6, compute
   `snapshots["upb_ratio"] = snapshots["current_upb"] / snapshots["orig_upb"]`, add `"upb_ratio"` to
   `numeric_features` (§8), and rerun §6–12. Look at `describe()` first: the guide (p. 10) says current UPB is
   rounded to the nearest $1,000 when the loan age is 6 months or less, and modified loans can carry deferred
   balances.
3. **Use one-hot encoding with one reference level dropped, for easier-to-read coefficients.**
   *Hint:* `OneHotEncoder(handle_unknown="ignore", min_frequency=20, drop="first")` in §8. Each remaining level's
   coefficient is then a difference from the dropped level (the first in sorted order, for example `status_t` `00`).
   Compare with the differences computed in §10. Read scikit-learn's documentation on how `drop` works together with
   `handle_unknown="ignore"` and `min_frequency` before trusting the result.
4. **Count prepayment as y = 0, and compare the event rates and AUC.** This is one of the outcome decisions waiting
   for Dr. Ku.
   *Hint:* in §7, change the line to `is_uncensored = snapshots["label"].isin(["event", "non_event", "censored_prepay"])`.
   `y` stays `label == "event"`, so prepaid loans become 0. Compare the new train/test event rates and ROC-AUC with
   the reference values (section 12 lists them), and think about which way you expect the event rate to move, and
   why.

> **Reminder: making a change official.** This notebook is the baseline, and `reference/` holds its official
> results; experimenting here changes neither. To make a change official, do it in a separate, reviewed step: edit the
> notebook (and `tests/test_baseline_no_leakage.py` if `build_snapshot()` changes), Restart and Run All with
> `SAVE_OUTPUTS = True`, check the differences that section 12 reports, copy the four files from `outputs/` into
> `reference/`, update `CLAUDE.md`, and run `uv run pytest tests -v`.